# Research data supporting "Efficient first-principles modeling of complex molecular crystals at sub-chemical accuracy"

This notebook accompanies our paper: **Efficient first-principles modeling of complex molecular crystals at sub-chemical accuracy**. It can be found on GitHub at https://github.com/benshi97/Data_LNOMBECC and explored interactively on [Colab](https://colab.research.google.com/github/benshi97/Data_LNOMBECC/blob/main/analyse.ipynb).

### Abstract

Molecules can form myriad crystalline polymorphs, each with distinct properties affecting their performance across diverse applications, from pharmaceuticals to functional materials and more.
Predicting the thermodynamically most stable polymorph from first principles remains a formidable challenge.
It requires methods that scale to large, technologically-relevant molecules while achieving very high accuracy (below 1 kJ/mol) on relative lattice energies.
Such accuracy – often termed sub-chemical accuracy – is generally beyond the reach of the workhorse density functional theory (DFT).
In this work, we introduce a framework combining several recent advances in correlated wavefunction theory (cWFT) to deliver accurate, cost-effective predictions of complex molecular crystals.
For 23 model organic molecules and 13 ice polymorphs, we predict crystal lattice energies to within experimental uncertainties at costs comparable to hybrid DFT, while being several orders of magnitude more efficient than previous cWFT approaches.
We extend this approach to a set of large, drug-like molecules including  axitinib and ROY – previously inaccessible to cWFT and where DFT is insufficient – achieving sub-chemical accuracy on the relative energies between challenging polymorphs.
With the reference data generated throughout this work, we have been able to further parametrize a DFT functional with unprecedented accuracy aligning with our predictions.
This cWFT framework as well as DFT functional are made openly available, providing new ranking tools to facilitate efficient high-throughput screening of molecular crystal polymorphs.


## Table of Contents

1. [*H*<sub>ads</sub> from the experimental literature](#*h*<sub>ads</sub>-from-the-experimental-literature)
2. [Computing *E*<sub>rlx</sub>, *E*<sub>therm</sub> and *E*<sub>ZPV</sub> from DFT](#computing-*e*<sub>rlx</sub>,-*e*<sub>therm</sub>-and-*e*<sub>zpv</sub>-from-dft)
3. [Computing *E*<sub>int</sub> with the SKZCAM protocol](#computing-*e*<sub>int</sub>-with-the-skzcam-protocol)
4. [Contributions for the cohesive and conformational energy from WFT in selected systems](#contributions-for-the-cohesive-and-conformational-energy-from-wft-in-selected-systems)
5. [Contribution for Dissociation of CH₃OH and H₂O Clusters on MgO(001)](#contribution-for-dissociation-of-ch₃oh-and-h₂o-clusters-on-mgo(001))
6. [Final *E*<sub>ads</sub> and *H*<sub>ads</sub> estimates from autoSKZCAM](#final-*e*<sub>ads</sub>-and-*h*<sub>ads</sub>-estimates-from-autoskzcam)
7. [Validating autoSKZCAM error bar estimates on CO<sub>2</sub> on MgO(001) system](#validating-autoskzcam-error-bar-estimates-on-co<sub>2</sub>-on-mgo(001)-system)
8. [Comparison between theory and/or experiment](#comparison-between-theory-and/or-experiment)
9. [Previous DFT literature](#previous-dft-literature)
10. [*E*<sub>int</sub> comparison between autoSKZCAM and DFT & RPA](#*e*<sub>int</sub>-comparison-between-autoskzcam-and-dft-&-rpa)
11. [Insights into molecular binding](#insights-into-molecular-binding)
12. [Comparison of accuracy and cost of DFT against autoSKZCAM](#comparison-of-accuracy-and-cost-of-dft-against-autoskzcam)
13. [Miscellaneous](#miscellaneous)



In [1]:
# Check if we are in Google Colab environment
try:
    import google.colab
    IN_COLAB = True
    usetex = False
except:
    import os
    IN_COLAB = False
    if os.path.expanduser('~') == '/Users/bshi':
        usetex = True
    else:
        usetex = False

if usetex:
    def textbf(text):
        return r'\textbf{' + text + '}'
else:
    def textbf(text):
        return text

# If in Google Colab, install the necessary data and set up the necessary environment
if IN_COLAB == True:
    repo_url = "https://github.com/benshi97/Data_LNOMBECC"

    # Clone the repository
    %cd /content
    !rm -rf /content/Data_LNOMBECC
    !git clone {repo_url}
    %cd /content/Data_LNOMBECC

replot_analysis = False

# Import necessary functions
from Scripts.analysis_scripts import *
from Scripts.plot_scripts import *

if usetex == True:
    textrue_import()
else:
    texfalse_import()

# Lattice energies of X23 dataset

## Experimental data analysis

In [2]:
# Collate the reference DMC and Experiment data
# The original data from Phys. Rev. Lett. 133, 046401 put hexamine and succinic acid at the end rather than alphabetically, which we will correct here
prl_names_order={idx+1: x for idx,x in enumerate(["1,4-cyclohexanedione" ,"Acetic Acid" ,"Adamantane" ,"Ammonia" ,"Anthracene" ,"Benzene" ,"CO$_2$" ,"Cyanamide" ,"Cytosine" ,"Ethyl carbamate" ,"Formamide" ,"Imidazole" ,"Naphthalene" ,r"Oxalic Acid $\alpha$" ,r"Oxalic Acid $\beta$" ,"Pyrazine" ,"Pyrazole" ,"Triazine" ,"Trioxane" ,"Uracil" ,"Urea" ,"Hexamine" ,"Succinic Acid" ])}
prl_to_x23_order = {idx+1: x for idx, x in enumerate([1,2,3,4,5,6,7,8,9,10,11,22,12,13,14,15,16,17,23,18,19,20,21])}
x23_labels = {x23_idx: prl_names_order[prl_to_x23_order[x23_idx]] for x23_idx in range(1,24)}


x23_dmc_data_raw=np.loadtxt('Data/X23/DMC/DMC.txt') # DMC estimates are in the PRL order
x23_dmc_elatt_dict = {x23_idx: list(x23_dmc_data_raw[prl_to_x23_order[x23_idx]-1]) for x23_idx in range(1,24)}

# Load the thermal corrections computed with vdW-DF2 MLIP from Chem. Sci., 2025,16, 11419-11433
pimd_deltah_raw =np.loadtxt('Data/X23/MLIP/DeltaH.txt',delimiter=',') # In PRL order
pimd_deltah_dict = {x23_idx: {'Elatt': pimd_deltah_raw[prl_to_x23_order[x23_idx]-1][0], 'DeltaH': -pimd_deltah_raw[prl_to_x23_order[x23_idx]-1][1], 'Etemp': -pimd_deltah_raw[prl_to_x23_order[x23_idx]-1][1] - pimd_deltah_raw[prl_to_x23_order[x23_idx]-1][0]} for x23_idx in range(1,24)}

x23_exp_enthalpy_dict = {}
x23_exp_elatt_dict = {}
# Load the experimental enthalpy data (this is also in the PRL order)
for x23_idx in range(1,24):
    system_enthalpy_data = -np.loadtxt(f'Data/X23/Experiment/expdata_{prl_to_x23_order[x23_idx]}_with_corrections.txt')
    if x23_labels[x23_idx] in ['Ammonia','Pyrazine',r'Oxalic Acid $\beta$']:
        system_enthalpy_data = system_enthalpy_data[:-1]
    x23_exp_enthalpy_dict[x23_idx] = system_enthalpy_data
    x23_exp_elatt_dict[x23_idx] = [x - pimd_deltah_dict[x23_idx]['Etemp'] for x in x23_exp_enthalpy_dict[x23_idx]]

### Correlation between standard deviation and adsorption enthalpy

In [3]:
# For a subset of systems, correlate the strength of the enthalpy and the standard deviation as well as range
x23_std_enthalpy_dict = {x23_idx: [np.std(x23_exp_enthalpy_dict[x23_idx]), np.mean(x23_exp_enthalpy_dict[x23_idx]), np.abs(np.max(x23_exp_enthalpy_dict[x23_idx]) - np.min(x23_exp_enthalpy_dict[x23_idx]))] for x23_idx in x23_exp_enthalpy_dict.keys() if len(x23_exp_enthalpy_dict[x23_idx]) > 4}

# Make a plot for the correlation between standard deviation and enthalpy as well as range and enthalpy
fig, axs = plt.subplots(1,2,figsize=(6.69,3.5),dpi=450, constrained_layout=True)
axs[0].scatter([x23_std_enthalpy_dict[x][1] for x in x23_std_enthalpy_dict.keys()], [x23_std_enthalpy_dict[x][0] for x in x23_std_enthalpy_dict.keys()], color=color_dict['black'])
# Fit a line and give the R^2 value
m, b = np.polyfit([x23_std_enthalpy_dict[x][1] for x in x23_std_enthalpy_dict.keys()], [x23_std_enthalpy_dict[x][0] for x in x23_std_enthalpy_dict.keys()], 1)
std_enthalpy_grad = m.copy()
std_enthalpy_int = b.copy()

correlation_matrix = np.corrcoef([x23_std_enthalpy_dict[x][1] for x in x23_std_enthalpy_dict.keys()], [x23_std_enthalpy_dict[x][0] for x in x23_std_enthalpy_dict.keys()])
correlation_xy = correlation_matrix[0,1]
r_squared = correlation_xy**2
axs[0].plot([x23_std_enthalpy_dict[x][1] for x in x23_std_enthalpy_dict.keys()], [m*x23_std_enthalpy_dict[x][1] + b for x in x23_std_enthalpy_dict.keys()], color=color_dict['red'], label = f'Best fit ($R^2={r_squared:.2f}$)')

for x in x23_std_enthalpy_dict.keys():
    axs[0].text(x23_std_enthalpy_dict[x][1], x23_std_enthalpy_dict[x][0] + 0.15, x23_labels[x], fontsize=6, ha='right', va='bottom')
axs[0].set_xlabel('Mean experimental enthalpy (kJ/mol)')
axs[0].set_ylabel('Standard deviation (kJ/mol)')
axs[0].text(0, 1.08, '(a)', transform=axs[0].transAxes, fontsize=10, verticalalignment='top')
axs[0].legend(fontsize=8)
axs[0].set_title('Std. dev. vs. enthalpy')



axs[1].scatter([x23_std_enthalpy_dict[x][1] for x in x23_std_enthalpy_dict.keys()], [x23_std_enthalpy_dict[x][2] for x in x23_std_enthalpy_dict.keys()], color=color_dict['black'])
for x in x23_std_enthalpy_dict.keys():
    axs[1].text(x23_std_enthalpy_dict[x][1], x23_std_enthalpy_dict[x][2] + 0.4, x23_labels[x], fontsize=6, ha='right', va='bottom')

# Fit a line and give the R^2 value
m, b = np.polyfit([x23_std_enthalpy_dict[x][1] for x in x23_std_enthalpy_dict.keys()], [x23_std_enthalpy_dict[x][2] for x in x23_std_enthalpy_dict.keys()], 1)

correlation_matrix = np.corrcoef([x23_std_enthalpy_dict[x][1] for x in x23_std_enthalpy_dict.keys()], [x23_std_enthalpy_dict[x][2] for x in x23_std_enthalpy_dict.keys()])
correlation_xy = correlation_matrix[0,1]
r_squared = correlation_xy**2
axs[1].plot([x23_std_enthalpy_dict[x][1] for x in x23_std_enthalpy_dict.keys()], [m*x23_std_enthalpy_dict[x][1] + b for x in x23_std_enthalpy_dict.keys()], color=color_dict['red'], label = f'Best fit ($R^2={r_squared:.2f}$)')
axs[1].text(0, 1.08, '(b)', transform=axs[1].transAxes, fontsize=10, verticalalignment='top')
axs[1].set_title('Range vs. enthalpy')


axs[1].set_xlabel('Mean experimental enthalpy (kJ/mol)')
axs[1].set_ylabel('Range (kJ/mol)')
axs[1].legend(fontsize=8)
plt.savefig('Figures/SI_Figure-X23_Exp_Enthalpy_Std_Range.png')


In [4]:
x23_exp_elatt_compile_dict = {}
for x23_idx in range(1,24):
    if len(x23_exp_enthalpy_dict[x23_idx]) > 4:
        enthalpy_range = np.abs(np.max(x23_exp_enthalpy_dict[x23_idx]) - np.min(x23_exp_enthalpy_dict[x23_idx]))
        enthalpy_sigma = np.std(x23_exp_enthalpy_dict[x23_idx])
        error_type = 'Real'
        error = 2*enthalpy_sigma
    else:
        enthalpy_range = np.nan
        enthalpy_sigma = np.nan
        error_type = 'Predicted'
        error = 2*(std_enthalpy_grad * np.mean(x23_exp_enthalpy_dict[x23_idx]) + std_enthalpy_int)
    x23_exp_elatt_compile_dict[x23_labels[x23_idx]] = {
        'Enthalpy measurements': ', '.join([f"{y:.1f}" for y in x23_exp_enthalpy_dict[x23_idx]]),
        'Etemp': f"{pimd_deltah_dict[x23_idx]['Etemp']:.1f}",
        'Mean enthalpy': f"{np.mean(x23_exp_enthalpy_dict[x23_idx]):.1f}",
        r"2sigma": f"{2*enthalpy_sigma:.1f}" if not np.isnan(enthalpy_sigma) else '-',
        r"Predicted 2sigma": f"{2*(std_enthalpy_grad * np.mean(x23_exp_enthalpy_dict[x23_idx]) + std_enthalpy_int):.1f}",
        "Range": f"{enthalpy_range:.1f}" if not np.isnan(enthalpy_range) else '-',
        'Lattice energy': f"{np.mean(x23_exp_elatt_dict[x23_idx]):.1f}",
        'Error': f"{error:.1f}",
        'Error type': error_type
}
x23_exp_elatt_df = pd.DataFrame.from_dict(x23_exp_elatt_compile_dict, orient='index')
display(x23_exp_elatt_df)

# Convert to LaTeX table
latex_input_str = '\n'.join(convert_df_to_latex_input(
    df = x23_exp_elatt_df,
    start_input = '\\begin{table}[h]\n',
    label = "tab:x23_experimental_lattice_energies",
    caption = r"Experimental lattice energies of the X23 set. All values are in kJ/mol.",
    replace_input= {
        "sigma": r"$\sigma$",
        "Etemp": r"$E_\text{temp}$~\cite{dellapiaAccurateEfficientMachine2025}",
        r" & \rotatebox{90}{Enthalpy measurements}": "Systems & Enthalpy measurements",
    },
    index = True,
    rotate_column_header=True,
    column_format="lp{40mm}rrrrrrrr",
    output_str=True,
    float_fmt="%.1f").splitlines()[6:-4]) + '\n'

with open('Tables/SI_Table-X23_Experimental_Lattice_Energies.tex', 'w') as f:
    f.write(r"""\LTcapwidth=\textwidth
    
\begin{longtable}{lp{35mm}rrrrrrrrr}
\caption{\label{tab:x23_experimental_lattice_energies}Compilation of experimental lattice energies of the X23 set, together with the mean and (analyzed) error estimate. All values are in kJ/mol.} \\

\toprule
Systems & Enthalpy measurements & \rotatebox{90}{$E_\text{temp}$~\cite{dellapiaAccurateEfficientMachine2025}} & \rotatebox{90}{Mean enthalpy} & \rotatebox{90}{2$\sigma$} & \rotatebox{90}{Predicted 2$\sigma$} & \rotatebox{90}{Range} & \rotatebox{90}{Lattice energy} & \rotatebox{90}{Error} & \rotatebox{90}{Error type} \\ 
\midrule
\endfirsthead



\caption[]{(continued)} \\
\endhead

\multicolumn{11}{r}{{Continued on next page}} \\
\endfoot

\bottomrule
\endlastfoot

""")
    f.write(latex_input_str)
    f.write(r"\end{longtable}")

,Enthalpy measurements,Etemp,Mean enthalpy,2sigma,Predicted 2sigma,Range,Lattice energy,Error,Error type
"1,4-cyclohexanedione","-84.2, -75.0, -84.0",6.4,-81.1,-,6.6,-,-87.5,6.6,Predicted
Acetic Acid,"-66.2, -68.7",5.0,-67.5,-,5.4,-,-72.4,5.4,Predicted
Adamantane,"-58.5, -59.6, -56.0, -55.3, -59.5, -60.2, -59....",6.7,-58.3,4.7,4.5,9.0,-64.9,4.7,Real
Ammonia,-31.1,5.8,-31.1,-,2.0,-,-36.8,2.0,Predicted
Anthracene,"-98.8, -98.5, -99.1, -96.2, -99.7, -103.5, -97...",5.6,-98.7,7.5,8.3,13.4,-104.2,7.5,Real
Benzene,"-41.5, -45.0, -45.1, -52.5, -48.0, -45.6, -43....",7.7,-45.1,5.4,3.3,11.0,-52.8,5.4,Real
CO$_2$,"-26.1, -24.7, -25.5, -25.5, -25.0",3.8,-25.3,1.0,1.5,1.4,-29.1,1.0,Real
Cyanamide,"-75.8, -75.0",4.2,-75.4,-,6.1,-,-79.6,6.1,Predicted
Cytosine,"-168.8, -155.3, -149.8, -155.0, -167.0, -176.0",5.3,-162.0,18.5,14.1,26.2,-167.2,18.5,Real
Ethyl carbamate,"-77.2, -72.3, -89.1, -76.0",6.2,-78.6,-,6.4,-,-84.9,6.4,Predicted


## Periodic HF analysis

In [5]:
# Analyse the periodic HF data
x23_hf_elatt_dict = {y: 0 for y in list(range(1,24))}
x23_hf_d4_elatt_dict = {y: 0 for y in list(range(1,24))}

x23_hf_elatt_contribution_dict = {y: {'VASP_Hard_Default':0,'VASP_Std_Default':0,'VASP_Std_High':0, 'Delta': 0, 'Final': 0} for y in list(range(1,24))}

for x23_idx in range(1,24):
    # Correct the order of the structures  
    molecule_ene = []
    solid_ene = []

    molecule = read(f'Data/X23/HF/VASP_Hard_Default/{prl_to_x23_order[x23_idx]:02d}/molecule/OUTCAR_HF.gz')
    solid = read(f'Data/X23/HF/VASP_Hard_Default/{prl_to_x23_order[x23_idx]:02d}/solid/OUTCAR_HF.gz')
    num_monomer = len(solid)/len(molecule)

    for calc_type in ['VASP_Hard_Default','VASP_Std_Default','VASP_Std_High']:
        molecule_ene += [get_vasp_energy(f'Data/X23/HF/{calc_type}/{prl_to_x23_order[x23_idx]:02d}/molecule/OUTCAR_HF.gz')]
        solid_ene += [get_vasp_energy(f'Data/X23/HF/{calc_type}/{prl_to_x23_order[x23_idx]:02d}/solid/OUTCAR_HF.gz')]
    
    elatt_energy = np.array(solid_ene)/num_monomer - np.array(molecule_ene)
    elatt = (elatt_energy[0] + elatt_energy[2] - elatt_energy[1])*1000
    x23_hf_elatt_dict[x23_idx] = elatt
    x23_hf_elatt_contribution_dict[x23_idx]['VASP_Hard_Default'] = (elatt_energy[0])*1000/kjmol
    x23_hf_elatt_contribution_dict[x23_idx]['VASP_Std_Default'] = (elatt_energy[1])*1000/kjmol
    x23_hf_elatt_contribution_dict[x23_idx]['VASP_Std_High'] = (elatt_energy[2])*1000/kjmol
    x23_hf_elatt_contribution_dict[x23_idx]['Delta'] = (elatt_energy[2] - elatt_energy[1])*1000/kjmol
    x23_hf_elatt_contribution_dict[x23_idx]['Final'] = elatt/kjmol

    molecule_d4 = np.loadtxt(f'Data/X23/HF/VASP_Hard_Default/{prl_to_x23_order[x23_idx]:02d}/molecule/D4_HF.out.gz')
    solid_d4 = np.loadtxt(f'Data/X23/HF/VASP_Hard_Default/{prl_to_x23_order[x23_idx]:02d}/solid/D4_HF.out.gz')
    x23_hf_d4_elatt_dict[x23_idx] = elatt + (solid_d4/num_monomer - molecule_d4)*Hartree*1000

# Convert x23_hf_elatt_contribution_dict to pandas dataframe
x23_hf_contribution_df = pd.DataFrame.from_dict(x23_hf_elatt_contribution_dict, orient='index')
x23_hf_contribution_df.index = [x23_labels[i] for i in x23_hf_contribution_df.index]
# Change the column names
x23_hf_contribution_df.columns = ['Hard (KSPACING=0.25)','Std (KSPACING=0.25)','Std (KSPACING=0.18)', r'$\Delta_k$', 'Final']
# Round to nearest integer
x23_hf_contribution_df = x23_hf_contribution_df.round(1)
display(x23_hf_contribution_df)
# Convert to latex table
convert_df_to_latex_input(
    df = x23_hf_contribution_df,
    start_input = '\\begin{table}[h]\n',
    label = "tab:x23_hf_contributions",
    caption = r"Contributions that make up the final HF lattice energies for the X23 set. All values are in kJ/mol.",
    replace_input= {
        "[t]": "[c]"
    },
    index = True,
    rotate_column_header=True,
    float_fmt="%.1f",
    filename = "Tables/SI_Table-X23_HF_Contributions.tex")


,Hard (KSPACING=0.25),Std (KSPACING=0.25),Std (KSPACING=0.18),$\Delta_k$,Final
"1,4-cyclohexanedione",-3.4,-5.7,-5.4,0.2,-3.2
Acetic Acid,-17.4,-20.1,-20.1,0.0,-17.4
Adamantane,41.5,41.5,41.4,-0.1,41.5
Ammonia,-9.1,-9.4,-9.4,-0.0,-9.1
Anthracene,54.9,55.0,55.0,-0.0,54.9
Benzene,23.0,23.0,23.0,-0.1,22.9
CO$_2$,-3.4,-4.3,-4.3,-0.0,-3.5
Cyanamide,-29.9,-30.7,-30.8,-0.1,-29.9
Cytosine,-48.0,-51.1,-51.1,0.0,-48.0
Ethyl carbamate,-17.4,-19.7,-19.8,-0.0,-17.4


### GPU costs

In [6]:
x23_hf_gpu_cost_dict = {y: {'VASP_Hard_Default': 0,'VASP_Std_Default': 0,'VASP_Std_High':0} for y in list(range(1,24))}

for x23_idx in range(1,24):

    for calc_type in ['VASP_Hard_Default','VASP_Std_Default','VASP_Std_High']:
        x23_hf_gpu_cost_dict[x23_idx][calc_type] += get_vasp_walltime(f'Data/X23/HF/{calc_type}/{prl_to_x23_order[x23_idx]:02d}/molecule/OUTCAR_HF.gz')*4/3600
        x23_hf_gpu_cost_dict[x23_idx][calc_type] += get_vasp_walltime(f'Data/X23/HF/{calc_type}/{prl_to_x23_order[x23_idx]:02d}/solid/OUTCAR_HF.gz')*4/3600

# Convert to dataframe
x23_hf_gpu_cost_df = pd.DataFrame.from_dict(x23_hf_gpu_cost_dict,orient='index')
x23_hf_gpu_cost_df.index = [f"{x23_labels[idx]}" for idx in x23_hf_gpu_cost_df.index]
# Add a row for the average
x23_hf_gpu_cost_df.loc['Average'] = x23_hf_gpu_cost_df.mean()
# Rename columns
x23_hf_gpu_cost_df.columns = ['Hard (0.25)','Std (0.25)','Std (0.18)']
# Round to integer
x23_hf_gpu_cost_df = x23_hf_gpu_cost_df.round(1)

display(x23_hf_gpu_cost_df)
# Convert to latex
convert_df_to_latex_input(
    df = x23_hf_gpu_cost_df,
    start_input = '\\begin{table}[h]\n',
    label = "tab:x23_hf_gpu_cost",
    caption = r"Estimated computational cost (in GPUh) for the composite VASP calculations used to compute the HF lattice energies of the X23 dataset on an NVIDIA A100 GPU. The three different settings correspond to different PAW potentials and plane-wave cutoffs (with $k$-point spacing given in parenthesis), as described in the text.",
    replace_input= {
        "[t]": "[c]"
    },
    index = True,
    rotate_column_header=True,
    float_fmt="%.1f",
    filename = "Tables/SI_Table-X23_Method_Costs_GPU.tex")

,Hard (0.25),Std (0.25),Std (0.18)
"1,4-cyclohexanedione",3.4,1.6,11.2
Acetic Acid,2.9,1.4,8.5
Adamantane,1.4,0.7,4.7
Ammonia,0.2,0.2,0.9
Anthracene,4.0,1.7,7.1
Benzene,2.4,0.9,4.5
CO$_2$,0.5,0.2,1.2
Cyanamide,4.3,2.0,17.3
Cytosine,4.2,1.7,10.4
Ethyl carbamate,2.6,1.4,15.0


## MBE cWFT analysis

In [7]:
# Calculating MP2, CCSD, CCSD(T) and CCSD(cT) results. This is given in the correct X23 order

x23_final_corr_elatt_dict = {y:{x:{} for x in ['MP2','CCSD','CCSD(T)','CCSD(cT)-fit']} for y in list(range(1,24))}
x23_final_corr_d4_elatt_dict = {y:{x:{} for x in ['MP2','CCSD','CCSD(T)','CCSD(cT)-fit']} for y in list(range(1,24))}
x23_final_cpuh_dict = {y: {'HF': 0, 'CCSD(T) corr.': 0} for y in list(range(1,24))}

method_basis = {
    '1B': { x: ' QZ' if x != 'MP2 Canonical' else ' CBS' for x in ['MP2 LAF', 'CCSD LAF', 'CCSD(T) LAF', 'CCSD(cT) LAF', 'MP2 Canonical'] },
    '2B': { x: ' TZ' if x != 'MP2 Canonical' else ' CBS' for x in ['MP2 LAF', 'CCSD LAF', 'CCSD(T) LAF', 'CCSD(cT) LAF', 'MP2 Canonical'] },
    '3B': { x: ' DZ' if x != 'MP2 Canonical' else ' CBS' for x in ['MP2 LAF', 'CCSD LAF', 'CCSD(T) LAF', 'CCSD(cT) LAF', 'MP2 Canonical'] }
}

method_basis['1B']['D4'] = ''
method_basis['2B']['D4'] = ''
method_basis['3B']['D4'] = ''

for x23_idx in range(1,24):
    system_all_methods_elatt_dict = {x:{} for x in ['MP2 LAF', 'CCSD LAF', 'CCSD(T) LAF', 'MP2 Canonical', 'MP2 Final', 'CCSD Final', 'CCSD(T) Final', 'CCSD(cT)-fit Final','D4']}
    system_total_time = 0
    for method in ['MP2 LAF', 'CCSD LAF', 'CCSD(T) LAF', 'MP2 Canonical','D4']:
        method_elatt_dict = {'1B': 0, '2B': 0, '3B': 0, 'Total': 0}
        with connect(f'Data/X23/cWFT/x23_{x23_idx:02d}_1b.db') as db1:
            monomer_ene_list = []
            for i in range(2):
                row = db1.get(id=i+1)
                monomer_ene_list += [row.data[f'{method}{method_basis["1B"][method]}']]
                if method == 'MP2 Canonical':
                    system_total_time += row.data['time']
                # print(row.data[f'{method}'])
            method_elatt_dict['1B'] = np.mean(monomer_ene_list[1:]) - monomer_ene_list[0]

        with connect(f'Data/X23/cWFT/x23_{x23_idx:02d}_2b.db') as db2:
            elatt_cwft = 0
            for i in range(len(db2)):
                row = db2.get(id=i+1)
                elatt_cwft += row.data[f'{method}{method_basis["2B"][method]}']*(row.data['count'])
                if method == 'MP2 Canonical':
                    system_total_time += row.data['time']
            method_elatt_dict['2B'] = elatt_cwft

        with connect(f'Data/X23/cWFT/x23_{x23_idx:02d}_3b.db') as db3:
            elatt_cwft = 0
            for i in range(len(db3)):
                row = db3.get(id=i+1)
                elatt_cwft += row.data[f'{method}{method_basis["3B"][method]}']*(row.data['count'])
                if method == 'MP2 Canonical':
                    system_total_time += row.data['time']
            method_elatt_dict['3B'] = elatt_cwft
        method_elatt_dict['Total'] = method_elatt_dict['1B'] + method_elatt_dict['2B'] + method_elatt_dict['3B']
        system_all_methods_elatt_dict[method] = method_elatt_dict.copy()
    
    for i in ['1B','2B','3B','Total']:
        system_all_methods_elatt_dict['MP2 Final'][i] = system_all_methods_elatt_dict['MP2 Canonical'][i]

        system_all_methods_elatt_dict['CCSD Final'][i] = system_all_methods_elatt_dict['CCSD LAF'][i] + system_all_methods_elatt_dict['MP2 Canonical'][i] - system_all_methods_elatt_dict['MP2 LAF'][i]

        system_all_methods_elatt_dict['CCSD(T) Final'][i] = system_all_methods_elatt_dict['CCSD(T) LAF'][i] + system_all_methods_elatt_dict['MP2 Canonical'][i] - system_all_methods_elatt_dict['MP2 LAF'][i]

        system_all_methods_elatt_dict['CCSD(cT)-fit Final'][i] = cT_calc(system_all_methods_elatt_dict['MP2 LAF'][i], system_all_methods_elatt_dict['CCSD LAF'][i], system_all_methods_elatt_dict['CCSD(T) LAF'][i]) + system_all_methods_elatt_dict['MP2 Canonical'][i] - system_all_methods_elatt_dict['MP2 LAF'][i]


    x23_final_corr_elatt_dict[x23_idx]['MP2'] = system_all_methods_elatt_dict['MP2 Final']
    x23_final_corr_elatt_dict[x23_idx]['CCSD'] = system_all_methods_elatt_dict['CCSD Final']
    x23_final_corr_elatt_dict[x23_idx]['CCSD(T)'] = system_all_methods_elatt_dict['CCSD(T) Final']
    x23_final_corr_elatt_dict[x23_idx]['CCSD(cT)-fit'] = system_all_methods_elatt_dict['CCSD(cT)-fit Final']
    for method in ['MP2','CCSD','CCSD(T)','CCSD(cT)-fit']:
        for x in ['1B','2B','3B','Total']:
            x23_final_corr_d4_elatt_dict[x23_idx][method][x] = x23_final_corr_elatt_dict[x23_idx][method][x] - system_all_methods_elatt_dict['D4'][x]

    x23_final_cpuh_dict[x23_idx]['CCSD(T) corr.'] = system_total_time*8/3600

### MP2, CCSD and CCSD(T) MBE contributions

In [8]:
# Plot the CCSD(T) MBE contributions for all X23 systems

x23_method_df = pd.DataFrame.from_dict({x23_labels[x23_idx]: {(method, mbe_type): x23_final_corr_elatt_dict[x23_idx][method][mbe_type] / kjmol  for method in ['MP2','CCSD','CCSD(T)'] for mbe_type in ['1B','2B','3B','Total']} for x23_idx in range(1,24)}, orient='index')

# Add row for mean signed difference compared to CCSD(T) and mean absolute difference
x23_method_df.loc['MSD'] = { (method, mbe_type): np.mean([x23_method_df.loc[x23_labels[x23_idx], (method, mbe_type)] - x23_method_df.loc[x23_labels[x23_idx], ('CCSD(T)', mbe_type)] for x23_idx in range(1,24)]) for method in ['MP2','CCSD','CCSD(T)'] for mbe_type in ['1B','2B','3B','Total']}
x23_method_df.loc['MAD'] = { (method, mbe_type): np.mean([np.abs(x23_method_df.loc[x23_labels[x23_idx], (method, mbe_type)] - x23_method_df.loc[x23_labels[x23_idx], ('CCSD(T)', mbe_type)]) for x23_idx in range(1,24)]) for method in ['MP2','CCSD','CCSD(T)'] for mbe_type in ['1B','2B','3B','Total']}


# Round to 1 d.p.
x23_method_df = x23_method_df.round(1)

display(x23_method_df)
# Convert to LaTeX table
convert_df_to_latex_input(
    df = x23_method_df,
    start_input = '\\begin{table}[h]\n',
    label = "tab:x23_mbe_contributions",
    caption = r"The MBE contributions to the lattice energy of the X23 set for several correlated wavefunction theory methods in the Composite `Full' basis approach. All values are in kJ/mol. We also report the mean signed deviation (MSD) and mean absolute deviation (MAD) of MP2 and CCSD compared to CCSD(T).",
    replace_input= {'DeltaMP2':r'Composite', "[t]": "[c]", r"{CCSD(T)} \\": r"{CCSD(T)} \\ \cmidrule(lr){2-5} \cmidrule(lr){6-9} \cmidrule(lr){10-13}"},
    float_fmt="%.1f",
    column_format="lrrrrrrrrrrrrrrrrrrrr",
    filename = "Tables/SI_Table-X23_MBE_Contributions.tex")


MP2                    CCSD                    CCSD(T)  \
                        1B     2B   3B  Total   1B     2B   3B  Total      1B   
1,4-cyclohexanedione  -6.4  -86.7  0.7  -92.5 -4.5  -71.3  3.0  -72.8    -5.8   
Acetic Acid           -4.8  -49.1  0.8  -53.1 -3.7  -42.1  2.2  -43.6    -4.5   
Adamantane            -0.4 -115.3  1.2 -114.4 -0.6  -94.5  4.3  -90.9    -1.0   
Ammonia               -0.9  -27.6  0.4  -28.1 -0.9  -24.2  1.3  -23.8    -1.1   
Anthracene            -2.4 -212.0  1.8 -212.5 -1.8 -141.0  4.4 -138.4    -2.7   
Benzene               -1.0  -92.7  2.1  -91.6 -0.9  -66.3  4.6  -62.6    -1.3   
CO$_2$                 0.1  -26.8  1.0  -25.7  0.1  -22.4  2.0  -20.2     0.1   
Cyanamide             -3.4  -59.6  2.2  -60.8 -2.1  -43.4  3.8  -41.7    -2.5   
Cytosine             -10.7 -113.7 -0.1 -124.4 -8.3  -88.5  3.3  -93.5   -10.3   
Ethyl carbamate       -2.9  -64.9  0.3  -67.4 -2.3  -56.1  2.1  -56.3    -3.0   
Formamide             -7.9  -41.8  0.2  -49.5 -6.0  -37.0  2.0  -41.0    -7.5   
Hexamine              -0.9 -111.7  1.0 -111.6 -0.5  -90.8  3.7  -87.6    -0.7   
Imidazole             -5.0  -84.6  1.1  -88.4 -3.6  -61.5  3.4  -61.6    -4.5   
Naphthalene           -1.4 -159.7  1.2 -159.9 -1.2 -109.0  2.8 -107.5    -1.8   
Oxalic Acid $\alpha$   1.8  -71.1  4.1  -65.3  1.5  -58.4  4.6  -52.2     2.4   
Oxalic Acid $\beta$   -3.5  -73.5  5.3  -71.7 -2.7  -59.8  4.9  -57.5    -2.7   
Pyrazine              -2.4  -96.3  1.8  -96.8 -1.8  -67.7  5.4  -64.0    -2.6   
Pyrazole              -4.0  -86.5  1.7  -88.8 -2.9  -62.9  3.8  -62.0    -3.7   
Succinic Acid         -7.7 -100.8  3.2 -105.3 -6.3  -83.8  6.0  -84.1    -7.5   
Triazine              -2.8  -71.0  2.2  -71.6 -2.0  -54.2  4.5  -51.8    -2.9   
Trioxane              -4.7  -67.7  1.5  -70.9 -3.9  -58.4  3.2  -59.0    -5.1   
Uracil               -12.3  -94.3 -0.5 -107.1 -8.8  -73.1  1.7  -80.1   -11.2   
Urea                  -5.8  -53.7 -2.6  -62.1 -3.9  -47.3  0.6  -50.6    -4.5   
MSD                   -0.2   -5.6 -2.6   -8.4  0.8   13.8 -0.6   14.0     0.0   
MAD                    0.4    6.9  2.6    8.5  0.8   13.8  0.6   14.0     0.0   

                                         
                         2B   3B  Total  
1,4-cyclohexanedione  -86.4  3.6  -88.6  
Acetic Acid           -50.5  2.6  -52.4  
Adamantane           -113.1  4.7 -109.3  
Ammonia               -28.7  1.4  -28.4  
Anthracene           -171.7  5.3 -169.1  
Benzene               -79.6  5.3  -75.7  
CO$_2$                -26.9  2.3  -24.5  
Cyanamide             -53.5  4.1  -51.9  
Cytosine             -107.5  3.8 -114.1  
Ethyl carbamate       -67.1  2.5  -67.6  
Formamide             -44.8  2.6  -49.7  
Hexamine             -108.9  4.0 -105.6  
Imidazole             -74.1  3.9  -74.6  
Naphthalene          -132.2  3.2 -130.8  
Oxalic Acid $\alpha$  -71.6  5.1  -64.1  
Oxalic Acid $\beta$   -74.3  6.5  -70.5  
Pyrazine              -81.7  6.4  -77.9  
Pyrazole              -75.6  4.3  -75.1  
Succinic Acid        -101.8  7.2 -102.2  
Triazine              -65.8  5.3  -63.4  
Trioxane              -68.8  3.9  -70.1  
Uracil                -90.1  2.7  -98.6  
Urea                  -56.9  0.3  -61.2  
MSD                     0.0  0.0    0.0  
MAD                     0.0  0.0    0.0

### CCSD(cT) MBE contributions

In [9]:
# Plot the CCSD(cT) MBE contributions for all X23 systems

x23_method_df = pd.DataFrame.from_dict({x23_labels[x23_idx]: {(method, mbe_type): x23_final_corr_elatt_dict[x23_idx][method][mbe_type] / kjmol  for method in ['CCSD(T)','CCSD(cT)-fit'] for mbe_type in ['1B','2B','3B','Total']} for x23_idx in range(1,24)}, orient='index')

# Add row for mean signed difference compared to CCSD(T) and mean absolute difference
x23_method_df.loc['MSD'] = { (method, mbe_type): np.mean([x23_method_df.loc[x23_labels[x23_idx], (method, mbe_type)] - x23_method_df.loc[x23_labels[x23_idx], ('CCSD(T)', mbe_type)] for x23_idx in range(1,24)]) for method in ['CCSD(T)','CCSD(cT)-fit'] for mbe_type in ['1B','2B','3B','Total']}
x23_method_df.loc['MAD'] = { (method, mbe_type): np.mean([np.abs(x23_method_df.loc[x23_labels[x23_idx], (method, mbe_type)] - x23_method_df.loc[x23_labels[x23_idx], ('CCSD(T)', mbe_type)]) for x23_idx in range(1,24)]) for method in ['CCSD(T)','CCSD(cT)-fit'] for mbe_type in ['1B','2B','3B','Total']}


# Round to 1 d.p.
x23_method_df = x23_method_df.round(1)

display(x23_method_df)
# Convert to LaTeX table
convert_df_to_latex_input(
    df = x23_method_df,
    start_input = '\\begin{table}[h]\n',
    label = "tab:x23_cT_mbe_contributions",
    caption = r"The MBE contributions to the lattice energy of the X23 set between CCSD(T) and CCSD(cT)-fit in the Composite `Full' basis approach. All values are in kJ/mol. We also report the mean signed deviation (MSD) and mean absolute deviation (MAD) compared to CCSD(T).",
    replace_input= {'DeltaMP2':r'Composite', "[t]": "[c]", r"{CCSD(cT)-fit} \\": r"{CCSD(cT)-fit} \\ \cmidrule(lr){2-5} \cmidrule(lr){6-9} \cmidrule(lr){10-13} \cmidrule(lr){14-17}"},
    float_fmt="%.1f",
    column_format="lrrrrrrrrrrrrrrrr",
    filename = "Tables/SI_Table-X23_cT_Contributions.tex")


CCSD(T)                    CCSD(cT)-fit              \
                          1B     2B   3B  Total           1B     2B   3B   
1,4-cyclohexanedione    -5.8  -86.4  3.6  -88.6         -5.6  -84.8  3.7   
Acetic Acid             -4.5  -50.5  2.6  -52.4         -4.4  -49.7  2.7   
Adamantane              -1.0 -113.1  4.7 -109.3         -1.0 -111.1  4.8   
Ammonia                 -1.1  -28.7  1.4  -28.4         -1.1  -28.3  1.5   
Anthracene              -2.7 -171.7  5.3 -169.1         -2.6 -166.6  5.4   
Benzene                 -1.3  -79.6  5.3  -75.7         -1.3  -77.7  5.4   
CO$_2$                   0.1  -26.9  2.3  -24.5          0.1  -26.4  2.4   
Cyanamide               -2.5  -53.5  4.1  -51.9         -2.4  -52.0  4.1   
Cytosine               -10.3 -107.5  3.8 -114.1        -10.1 -105.2  3.9   
Ethyl carbamate         -3.0  -67.1  2.5  -67.6         -2.9  -66.1  2.6   
Formamide               -7.5  -44.8  2.6  -49.7         -7.3  -44.1  2.7   
Hexamine                -0.7 -108.9  4.0 -105.6         -0.6 -107.0  4.0   
Imidazole               -4.5  -74.1  3.9  -74.6         -4.3  -72.3  4.0   
Naphthalene             -1.8 -132.2  3.2 -130.8         -1.7 -128.5  3.3   
Oxalic Acid $\alpha$     2.4  -71.6  5.1  -64.1          2.3  -70.1  5.1   
Oxalic Acid $\beta$     -2.7  -74.3  6.5  -70.5         -2.7  -72.7  6.4   
Pyrazine                -2.6  -81.7  6.4  -77.9         -2.5  -79.6  6.5   
Pyrazole                -3.7  -75.6  4.3  -75.1         -3.6  -73.8  4.3   
Succinic Acid           -7.5 -101.8  7.2 -102.2         -7.4  -99.9  7.3   
Triazine                -2.9  -65.8  5.3  -63.4         -2.8  -64.4  5.4   
Trioxane                -5.1  -68.8  3.9  -70.1         -5.0  -67.9  3.9   
Uracil                 -11.2  -90.1  2.7  -98.6        -10.8  -88.0  2.9   
Urea                    -4.5  -56.9  0.3  -61.2         -4.4  -56.1  0.5   
MSD                      0.0    0.0  0.0    0.0          0.1    1.7  0.1   
MAD                      0.0    0.0  0.0    0.0          0.1    1.7  0.1   

                             
                      Total  
1,4-cyclohexanedione  -86.7  
Acetic Acid           -51.5  
Adamantane           -107.2  
Ammonia               -27.9  
Anthracene           -163.8  
Benzene               -73.6  
CO$_2$                -24.0  
Cyanamide             -50.2  
Cytosine             -111.3  
Ethyl carbamate       -66.4  
Formamide             -48.8  
Hexamine             -103.5  
Imidazole             -72.6  
Naphthalene          -127.0  
Oxalic Acid $\alpha$  -62.7  
Oxalic Acid $\beta$   -69.0  
Pyrazine              -75.6  
Pyrazole              -73.0  
Succinic Acid        -100.1  
Triazine              -61.8  
Trioxane              -69.0  
Uracil                -96.1  
Urea                  -60.0  
MSD                     1.9  
MAD                     1.9

### Subtractive embedding with D4 dispersion

In [10]:
x23_final_elatt_dict = {y:{x:{} for x in ['MP2','CCSD','CCSD(T)','CCSD(cT)-fit']} for y in list(range(1,24))}
x23_final_d4_elatt_dict = {y:{x:{} for x in ['MP2','CCSD','CCSD(T)','CCSD(cT)-fit']} for y in list(range(1,24))}

mad_list = []
rmsd_list = []

for method in ['MP2','CCSD','CCSD(T)','CCSD(cT)-fit']:
    for x23_idx in range(1,24):
        x23_final_elatt_dict[x23_idx][method] = (x23_hf_elatt_dict[x23_idx] + x23_final_corr_elatt_dict[x23_idx][method]['Total'])/kjmol
        x23_final_d4_elatt_dict[x23_idx][method] = (x23_hf_d4_elatt_dict[x23_idx] + x23_final_corr_d4_elatt_dict[x23_idx][method]['Total'])/kjmol
        if method == 'CCSD(T)':
            mad_list += [abs(x23_final_elatt_dict[x23_idx][method] - x23_final_d4_elatt_dict[x23_idx][method])]
            rmsd_list += [(x23_final_elatt_dict[x23_idx][method] - x23_final_d4_elatt_dict[x23_idx][method])**2]

mad = np.mean(mad_list)


# Create a DataFrame to compare the CCSD(T) estimates with and without D4 correction
x23_d4_compare_dict = {x23_labels[x23_idx]: {'Without D4': x23_final_elatt_dict[x23_idx]['CCSD(T)'], 'With D4': x23_final_d4_elatt_dict[x23_idx]['CCSD(T)'], 'Difference': x23_final_elatt_dict[x23_idx]['CCSD(T)'] - x23_final_d4_elatt_dict[x23_idx]['CCSD(T)']}  for x23_idx in range(1,24)}
x23_d4_compare_dict['Mean'] = {'Without D4': '-', 'With D4': '-', 'Difference': mad}

x23_d4_compare_df = pd.DataFrame.from_dict(x23_d4_compare_dict, orient='index')
# Round to 1 decimal place using applymap
x23_d4_compare_df = x23_d4_compare_df.map(
    lambda x: round(x, 1) if isinstance(x, (int, float)) else x
)
# Convert to latex
convert_df_to_latex_input(
    df = x23_d4_compare_df,
    start_input = '\\begin{table}[h]\n',
    label = "tab:x23_d4_compare",
    caption = r"Lattice energy (in kJ/mol) for the X23 dataset when LNO-MBE-CCSD(T) is corrected with (and without) the D4 dispersion correction.",
    replace_input= {
        "[t]": "[c]"
    },
    index = True,
    filename = "Tables/SI_Table-X23_D4_Compare.tex",
    float_fmt = r"%.1f",)

display(x23_d4_compare_df)

,Without D4,With D4,Difference
"1,4-cyclohexanedione",-91.8,-92.2,0.4
Acetic Acid,-69.8,-70.3,0.5
Adamantane,-67.9,-67.5,-0.3
Ammonia,-37.5,-38.6,1.1
Anthracene,-114.2,-113.9,-0.3
Benzene,-52.8,-52.8,0.0
CO$_2$,-27.9,-27.9,-0.0
Cyanamide,-81.9,-82.4,0.5
Cytosine,-162.1,-161.9,-0.2
Ethyl carbamate,-85.0,-85.0,0.1


## Comparisons to experimental and theoretical literature

In [11]:
# Make a table which compares CCSD(T), CCSD(cT), DMC and experimental values
x23_final_compare_dict = {x23_labels[x23_idx]: {'CCSD(T)': x23_final_elatt_dict[x23_idx]['CCSD(T)'], 'CCSD(cT)-fit': x23_final_elatt_dict[x23_idx]['CCSD(cT)-fit'], 'DMC': x23_dmc_elatt_dict[x23_idx][0], 'DMC Error': x23_dmc_elatt_dict[x23_idx][1], 'Expt': x23_exp_elatt_compile_dict[x23_labels[x23_idx]]['Lattice energy'], 'Expt Error': x23_exp_elatt_compile_dict[x23_labels[x23_idx]]['Error']}  for x23_idx in range(1,24)}

# Save the dictionary as npy file
np.save('Data/X23/cWFT/X23_Final_Data.npy', x23_final_compare_dict)

x23_final_compare_df = pd.DataFrame.from_dict(x23_final_compare_dict, orient='index')

refs = {
    "MAD [CCSD(T)]": "CCSD(T)",
    "MAD (Expt)": "Expt",
    "MAD (DMC)": "DMC"
}

for label, ref_col in refs.items():
    row = {}
    for col in ["CCSD(T)", "CCSD(cT)-fit", "DMC", "Expt"]:
        if col == ref_col:
            row[col] = 0.0
        else:
            row[col] = np.mean([
                abs(
                    float(x23_final_compare_df.loc[x23_labels[idx], col]) -
                    float(x23_final_compare_df.loc[x23_labels[idx], ref_col])
                )
                for idx in range(1, 24)
            ])
    # constant values
    row["DMC Error"] = "-"
    row["Expt Error"] = "-"
    
    # assign row
    x23_final_compare_df.loc[label] = row

# Add row for MSD w.r.t. DMC
x23_final_compare_df.loc['MSD (DMC)'] = {
    'CCSD(T)': np.mean([x23_final_compare_df.loc[x23_labels[x23_idx], 'CCSD(T)'] - x23_final_compare_df.loc[x23_labels[x23_idx], 'DMC'] for x23_idx in range(1,24)]),
    'CCSD(cT)-fit': np.mean([x23_final_compare_df.loc[x23_labels[x23_idx], 'CCSD(cT)-fit'] - x23_final_compare_df.loc[x23_labels[x23_idx], 'DMC'] for x23_idx in range(1,24)]),
    'DMC': 0.0,
    'DMC Error': '-',
    'Expt': np.mean([float(x23_final_compare_df.loc[x23_labels[x23_idx], 'Expt']) - x23_final_compare_df.loc[x23_labels[x23_idx], 'DMC'] for x23_idx in range(1,24)]),
    'Expt Error': '-'
}


# Print the number of systems which are within 2 kJ/mol between DMC and CCSD(T), including the DMC error bars
print("Number of systems within 2 kJ/mol between DMC and CCSD(T):", sum([
    abs(x23_final_compare_df.loc[x23_labels[x23_idx], 'CCSD(T)'] - x23_final_compare_df.loc[x23_labels[x23_idx], 'DMC']) <= x23_final_compare_df.loc[x23_labels[x23_idx], 'DMC Error'] + 2
    for x23_idx in range(1, 24)
]))



# Round to 1 decimal place
x23_final_compare_df = x23_final_compare_df.map(
    lambda x: round(x, 1) if isinstance(x, (int, float)) else x
)
display(x23_final_compare_df)

# Convert to latex
convert_df_to_latex_input(
    df = x23_final_compare_df,
    start_input = '\\begin{table}[h]\n',
    label = "tab:x23_final_compare",
    caption = r"Final lattice energy (in kJ/mol) for the X23 dataset computed with different methods, compared to experimental values. The DMC values are taken from Ref.~\citenum{dellapiaHowAccurateAre2024}. The mean absolute deviation (MAD) is also reported with respect to (LNO-MBE-)CCSD(T), experiment and DMC.",
    replace_input= {},
    index = True,
    float_fmt = r"%.1f",
    filename = "Tables/SI_Table-X23_Final_Compare.tex",
    column_format="lrrrrrr"
)

Number of systems within 2 kJ/mol between DMC and CCSD(T): 15


,CCSD(T),CCSD(cT)-fit,DMC,DMC Error,Expt,Expt Error
"1,4-cyclohexanedione",-91.8,-89.9,-88.3,1.0,-87.5,6.6
Acetic Acid,-69.8,-68.9,-71.7,0.6,-72.4,5.4
Adamantane,-67.9,-65.8,-61.0,2.3,-64.9,4.7
Ammonia,-37.5,-37.0,-38.2,0.1,-36.8,2.0
Anthracene,-114.2,-108.9,-100.2,0.5,-104.2,7.5
Benzene,-52.8,-50.7,-49.8,0.2,-52.8,5.4
CO$_2$,-27.9,-27.4,-29.4,0.2,-29.1,1.0
Cyanamide,-81.9,-80.2,-83.6,0.4,-79.6,6.1
Cytosine,-162.1,-159.3,-156.2,1.0,-167.2,18.5
Ethyl carbamate,-85.0,-83.8,-84.2,1.3,-84.9,6.4


In [12]:
# Plot comparing CCSD(T) and DMC lattice energies
fig, ax = plt.subplots(figsize=(6.65,6), dpi = 450, constrained_layout=True)
ax.scatter([x23_final_elatt_dict[x23_idx]['CCSD(T)'] for x23_idx in range(1,24)], [x23_dmc_elatt_dict[x23_idx][0] for x23_idx in range(1,24)], color=color_dict['black'])
for x23_idx in range(1,24):
    if x23_idx in [13]:
        ax.text(x23_final_elatt_dict[x23_idx]['CCSD(T)'] + -5, x23_dmc_elatt_dict[x23_idx][0] -3.0, x23_labels[x23_idx], fontsize=8, ha='right', va='bottom')
    elif x23_idx in [17]:
        ax.text(x23_final_elatt_dict[x23_idx]['CCSD(T)'] + - 6, x23_dmc_elatt_dict[x23_idx][0] - 3.0, x23_labels[x23_idx], fontsize=8, ha='right', va='bottom')
    elif x23_idx in [15]:
        ax.text(x23_final_elatt_dict[x23_idx]['CCSD(T)'] + 4, x23_dmc_elatt_dict[x23_idx][0] - 2.5, x23_labels[x23_idx], fontsize=8, ha='left', va='bottom')
    elif x23_idx in [21]:
        ax.text(x23_final_elatt_dict[x23_idx]['CCSD(T)'] + 4, x23_dmc_elatt_dict[x23_idx][0] - 3.0, x23_labels[x23_idx], fontsize=8, ha='left', va='bottom')
    elif x23_idx in [1,16,3,18,8]:
        ax.text(x23_final_elatt_dict[x23_idx]['CCSD(T)'] + 4, x23_dmc_elatt_dict[x23_idx][0] - 1.0, x23_labels[x23_idx], fontsize=8, ha='left', va='bottom')
    else:
        ax.text(x23_final_elatt_dict[x23_idx]['CCSD(T)'] - 4, x23_dmc_elatt_dict[x23_idx][0] - 1.0, x23_labels[x23_idx], fontsize=8, ha='right', va='bottom')

# Fit a line and give the R^2 value
m, b = np.polyfit([x23_final_elatt_dict[x23_idx]['CCSD(T)'] for x23_idx in range(1,24)], [x23_dmc_elatt_dict[x23_idx][0] for x23_idx in range(1,24)], 1)
correlation_matrix = np.corrcoef([x23_final_elatt_dict[x23_idx]['CCSD(T)'] for x23_idx in range(1,24)], [x23_dmc_elatt_dict[x23_idx][0] for x23_idx in range(1,24)])
correlation_xy = correlation_matrix[0,1]
r_squared = correlation_xy**2
ax.plot([x23_final_elatt_dict[x23_idx]['CCSD(T)'] for x23_idx in range(1,24)], [m*x23_final_elatt_dict[x23_idx]['CCSD(T)'] + b for x23_idx in range(1,24)], color=color_dict['red'], label = f'Best fit ($R^2={r_squared:.2f}$)')
ax.plot([-200,0],[-200,0], color=color_dict['blue'], linestyle='--', label='y=x', lw=2)
ax.set_xlabel('LNO-MBE-CCSD(T) lattice energy (kJ/mol)')
ax.set_ylabel('DMC lattice energy (kJ/mol)')
ax.set_title('CCSD(T) vs. DMC lattice energies')
ax.legend(fontsize=8)
ax.set_ylim([-200,0])
ax.set_xlim([-200,0])
plt.savefig('Figures/SI_Figure-X23_CCSDT_vs_DMC_Lattice_Energies.png')

In [13]:
multilevel_syty_references = {
    '1,4-cyclohexanedione': -94.4,
    'Acetic Acid': -71.2,
    'Adamantane': -66.0,
    'Ammonia': -37.6,
    'Anthracene': -111.4,
    'Benzene': -51.6,
    'CO$_2$': -29.4,
    'Cyanamide': -82.6,
    'Cytosine': -163.1,
    'Ethyl carbamate': -86.5,
    'Formamide': -84.0,
    'Hexamine': -88.0,
    'Imidazole': -88.4,
    'Naphthalene': -82.5,
    'Oxalic Acid $\\alpha$': -103.0,
    'Oxalic Acid $\\beta$': -101.6,
    'Pyrazine': -64.3,
    'Pyrazole': -79.3,
    'Succinic Acid': -128.2,
    'Triazine': -60.3,
    'Trioxane': -67.8,
    'Uracil': -139.7,
    'Urea': -111.0,
}
rpa_gwse_references = {
    'Adamantane': -69.4,
    'Anthracene': -112.7,
    'Naphthalene': -81.7,
    'Benzene': -55.3,
    'CO$_2$': -28.4,
    'Urea': -102.5,
    'Ammonia': -37.2,
    'Cyanamide': -79.7,
    'Oxalic Acid $\\alpha$': -96.3,
    'Oxalic Acid $\\beta$': -96.1,
}

hmbi_references = {
    'Formamide': -80.4,
    'Imidazole': -90.8,
    'Benzene': -54.0,
    'Ammonia': -40.2,
    'CO$_2$': -29.5,
}

multilevel_sherrill_references = {
    'Benzene': {
        '2B': -57.99,
        '3B': 3.63,
    },
    'CO$_2$': {
        '2B': -30.11,
        '3B': 1.43,
    },
    'Triazine': {
        '2B': -58.36,
        '3B': 4.75,
    },
}

cervinka_ccsdt_mbe_3b = {
    'Ammonia': -39.1,
    'CO$_2$': -21.2,
    'Formamide': -89.9,
    'Acetic Acid': -70.6,
}

x23_1b_cc_energies = {}

for x23_idx in range(1,24):
    with connect(f'Data/X23/cWFT/x23_{x23_idx:02d}_1b.db') as db1:
        monomer_ene_list = []
        for i in range(2):
            row = db1.get(id=i+1)
            monomer_ene_list += [row.data[f'HF CBS'] + row.data[f'MP2 Canonical CBS'] + row.data[f'CCSD(T) LAF QZ'] - row.data[f'MP2 LAF QZ']]
            # print(row.data[f'{method}'])
        x23_1b_cc_energies[x23_labels[x23_idx]] = (np.mean(monomer_ene_list[1:]) - monomer_ene_list[0]) / kjmol

# Add 1B and total to multilevel_sherrill_references
for system in multilevel_sherrill_references.keys():
    multilevel_sherrill_references[system]['1B'] = x23_1b_cc_energies[system]
    multilevel_sherrill_references[system]['Total'] = multilevel_sherrill_references[system]['1B'] + multilevel_sherrill_references[system]['2B'] + multilevel_sherrill_references[system]['3B']

# Make a DataFrame of all of the reference data
x23_reference_df = pd.DataFrame.from_dict({
    r'\shortstack{Multilevel\\Syty \etal{}~\cite{sytyMultiLevelCoupledClusterDescription2025}}': multilevel_syty_references,
    r'\shortstack{RPA+GWSE\\(Klime\v{s}~\cite{klimesLatticeEnergiesMolecular2016})}': rpa_gwse_references,
    r'\shortstack{HMBI CCSD(T)\\1B+2B+FF($>$2B)\\(Wen and Beran~\cite{wenAccurateMolecularCrystal2011a})}': hmbi_references,
    r'\shortstack{CCSD(T)\\1B+2B+3B MBE\\(Sherrill \etal{}~\cite{sargentBenchmarkingTwobodyContributions2023a,xieAssessmentThreebodyDispersion2023a})}': {system: multilevel_sherrill_references[system]['Total'] for system in multilevel_sherrill_references.keys()},
    r'\shortstack{CCSD(T)\\1B+2B+3B+ELST($>$3B) MBE\\(\v{C}ervinka~\cite{cervinkaCCSDTCBSFragmentbased2016})}': cervinka_ccsdt_mbe_3b
}, orient='index').T


x23_reference_df = x23_reference_df.fillna('-')

# Add MAD against CCSD(T), DMC and Experiment from x23_final_compare_df
for label, ref_col in refs.items():
    row = {}
    for col in x23_reference_df.columns:
        if col == ref_col:
            row[col] = 0.0
        else:
            row[col] = np.mean([
                abs(
                    float(x23_reference_df.loc[x23_labels[idx], col]) -
                    float(x23_final_compare_df.loc[x23_labels[idx], ref_col])
                )
                for idx in range(1, 24)
                if x23_reference_df.loc[x23_labels[idx], col] != '-' and x23_final_compare_df.loc[x23_labels[idx], ref_col] != '-'
            ])
    # assign row
    x23_reference_df.loc[label] = row

# Round to 1 decimal place
x23_reference_df = x23_reference_df.map(
    lambda x: round(x, 1) if isinstance(x, (int, float)) else x
)

display(x23_reference_df)

# Convert to latex
convert_df_to_latex_input(
    df = x23_reference_df,
    start_input = '\\begin{table}[h]\n',
    label = "tab:x23_references_compare",
    caption = r"Comparison of reference literature lattice energies (in kJ/mol) for X23. The mean absolute deviation (MAD) is reported with respect to LNO-MBE-CCSD(T), experiment and DMC~\cite{dellapiaHowAccurateAre2024}.",
    replace_input= {},
    index = True,
    float_fmt = r"%.1f",
    rotate_column_header=True,
    filename = "Tables/SI_Table-X23_References_Compare.tex",
    column_format=r"lrrrrr"
)


,\shortstack{Multilevel\\Syty \etal{}~\cite{sytyMultiLevelCoupledClusterDescription2025}},\shortstack{CCSD(T)\\1B+2B+3B+ELST($>$3B) MBE\\(\v{C}ervinka~\cite{cervinkaCCSDTCBSFragmentbased2016})},\shortstack{RPA+GWSE\\(Klime\v{s}~\cite{klimesLatticeEnergiesMolecular2016})},\shortstack{HMBI CCSD(T)\\1B+2B+FF($>$2B)\\(Wen and Beran~\cite{wenAccurateMolecularCrystal2011a})},"\shortstack{CCSD(T)\\1B+2B+3B MBE\\(Sherrill \etal{}~\cite{sargentBenchmarkingTwobodyContributions2023a,xieAssessmentThreebodyDispersion2023a})}"
"1,4-cyclohexanedione",-94.4,-,-,-,-
Acetic Acid,-71.2,-70.6,-,-,-
Adamantane,-66.0,-,-69.4,-,-
Ammonia,-37.6,-39.1,-37.2,-40.2,-
Anthracene,-111.4,-,-112.7,-,-
Benzene,-51.6,-,-55.3,-54.0,-53.8
CO$_2$,-29.4,-21.2,-28.4,-29.5,-28.7
Cyanamide,-82.6,-,-79.7,-,-
Cytosine,-163.1,-,-,-,-
Ethyl carbamate,-86.5,-,-,-,-


### H-bonded subset

In [14]:
# Get MAD for H-bonded subsets
h_bonded_x23_labels = ["Acetic Acid",
 "Ammonia",
 "Cyanamide",
 "Ethyl carbamate",
 "Formamide",
 "Urea",
 "Succinic Acid",
 r"Oxalic Acid $\alpha$",
 r"Oxalic Acid $\beta$"]

x23_h_bonded_dict = {x23_labels[x23_idx]: x23_final_compare_df.loc[x23_labels[x23_idx]] for x23_idx in range(1,24) if x23_labels[x23_idx] in h_bonded_x23_labels}

# Add rows for MAD
for label, ref_col in refs.items():
    row = {}
    for col in ["CCSD(T)", "CCSD(cT)-fit", "DMC", "Expt"]:
        if col == ref_col:
            row[col] = 0.0
        else:
            row[col] = np.mean([
                abs(
                    float(x23_h_bonded_dict[x23_labels[idx]][col]) -
                    float(x23_h_bonded_dict[x23_labels[idx]][ref_col])
                )
                for idx in range(1, 24) if x23_labels[idx] in h_bonded_x23_labels
            ])
    # constant values
    row["DMC Error"] = "-"
    row["Expt Error"] = "-"
    
    # assign row
    x23_h_bonded_dict[label] = row

# Add row for MSD w.r.t. DMC
x23_h_bonded_dict['MSD (DMC)'] = {
    'CCSD(T)': np.mean([x23_h_bonded_dict[x23_labels[x23_idx]]['CCSD(T)'] - x23_h_bonded_dict[x23_labels[x23_idx]]['DMC'] for x23_idx in range(1,24) if x23_labels[x23_idx] in h_bonded_x23_labels]),
    'CCSD(cT)-fit': np.mean([x23_h_bonded_dict[x23_labels[x23_idx]]['CCSD(cT)-fit'] - x23_h_bonded_dict[x23_labels[x23_idx]]['DMC'] for x23_idx in range(1,24) if x23_labels[x23_idx] in h_bonded_x23_labels]),
    'DMC': 0.0,
    'DMC Error': '-',
    'Expt': np.mean([float(x23_h_bonded_dict[x23_labels[x23_idx]]['Expt']) - x23_h_bonded_dict[x23_labels[x23_idx]]['DMC'] for x23_idx in range(1,24) if x23_labels[x23_idx] in h_bonded_x23_labels]),
    'Expt Error': '-'
}

x23_h_bonded_df = pd.DataFrame.from_dict(x23_h_bonded_dict, orient='index')

# Round to 1 decimal place
x23_h_bonded_df = x23_h_bonded_df.map(
    lambda x: round(x, 1) if isinstance(x, (int, float)) else x
)

display(x23_h_bonded_df)

# Convert to latex
convert_df_to_latex_input(
    df = x23_h_bonded_df,
    start_input = '\\begin{table}[h]\n',
    label = "tab:x23_h_bonded_compare",
    caption = r"Lattice energy (in kJ/mol) for the H-bonded subset of the X23 dataset computed with different methods, compared to experimental values. The DMC values are taken from Ref.~\citenum{dellapiaHowAccurateAre2024}. The mean absolute deviation (MAD) is also reported with respect to (LNO-MBE-)CCSD(T), experiment and DMC.",
    replace_input= {},
    index = True,
    float_fmt = r"%.1f",
    filename = "Tables/SI_Table-X23_H_Bonded_Compare.tex",
    column_format="lrrrrrr"
)


,CCSD(T),CCSD(cT)-fit,DMC,DMC Error,Expt,Expt Error
Acetic Acid,-69.8,-68.9,-71.7,0.6,-72.4,5.4
Ammonia,-37.5,-37.0,-38.2,0.1,-36.8,2.0
Cyanamide,-81.9,-80.2,-83.6,0.4,-79.6,6.1
Ethyl carbamate,-85.0,-83.8,-84.2,1.3,-84.9,6.4
Formamide,-80.1,-79.2,-81.0,1.0,-78.6,5.8
Oxalic Acid $\alpha$,-99.4,-98.0,-102.6,1.4,-96.4,7.8
Oxalic Acid $\beta$,-98.7,-97.2,-102.3,0.6,-94.6,7.8
Succinic Acid,-124.9,-122.8,-125.2,0.5,-116.8,6.0
Urea,-106.8,-105.6,-108.5,0.3,-99.2,7.0
MAD [CCSD(T)],0.0,1.3,1.6,-,3.3,-


### vdW-bonded subset

In [15]:
vdw_bonded_x23_labels = ["1,4-cyclohexanedione",
 "Adamantane",
 "Anthracene",
 "Benzene",
 "Hexamine",
 "Naphthalene",
 "Pyrazine",
 "Pyrazole",
 "Triazine",
 "Trioxane",
 "CO$_2$"]

x23_vdw_bonded_dict = {x23_labels[x23_idx]: x23_final_compare_df.loc[x23_labels[x23_idx]] for x23_idx in range(1,24) if x23_labels[x23_idx] in vdw_bonded_x23_labels}

# Add rows for MAD
for label, ref_col in refs.items():
    row = {}
    for col in ["CCSD(T)", "CCSD(cT)-fit", "DMC", "Expt"]:
        if col == ref_col:
            row[col] = 0.0
        else:
            row[col] = np.mean([
                abs(
                    float(x23_vdw_bonded_dict[x23_labels[idx]][col]) -
                    float(x23_vdw_bonded_dict[x23_labels[idx]][ref_col])
                )
                for idx in range(1, 24) if x23_labels[idx] in vdw_bonded_x23_labels
            ])
    # constant values
    row["DMC Error"] = "-"
    row["Expt Error"] = "-"
    
    # assign row
    x23_vdw_bonded_dict[label] = row
# Add row for MSD w.r.t. DMC
x23_vdw_bonded_dict['MSD (DMC)'] = {
    'CCSD(T)': np.mean([x23_vdw_bonded_dict[x23_labels[x23_idx]]['CCSD(T)'] - x23_vdw_bonded_dict[x23_labels[x23_idx]]['DMC'] for x23_idx in range(1,24) if x23_labels[x23_idx] in vdw_bonded_x23_labels]),
    'CCSD(cT)-fit': np.mean([x23_vdw_bonded_dict[x23_labels[x23_idx]]['CCSD(cT)-fit'] - x23_vdw_bonded_dict[x23_labels[x23_idx]]['DMC'] for x23_idx in range(1,24) if x23_labels[x23_idx] in vdw_bonded_x23_labels]),
    'DMC': 0.0,
    'DMC Error': '-',
    'Expt': np.mean([float(x23_vdw_bonded_dict[x23_labels[x23_idx]]['Expt']) - x23_vdw_bonded_dict[x23_labels[x23_idx]]['DMC'] for x23_idx in range(1,24) if x23_labels[x23_idx] in vdw_bonded_x23_labels]),
    'Expt Error': '-'
}
x23_vdw_bonded_df = pd.DataFrame.from_dict(x23_vdw_bonded_dict, orient='index')
# Round to 1 decimal place
x23_vdw_bonded_df = x23_vdw_bonded_df.map(
    lambda x: round(x, 1) if isinstance(x, (int, float)) else x
)
display(x23_vdw_bonded_df)

# Convert to latex
convert_df_to_latex_input(
    df = x23_vdw_bonded_df,
    start_input = '\\begin{table}[h]\n',
    label = "tab:x23_vdw_bonded_compare",
    caption = r"Lattice energy (in kJ/mol) for the vdW-bonded subset of the X23 dataset computed with different methods, compared to experimental values. The DMC values are taken from Ref.~\citenum{dellapiaHowAccurateAre2024}. The mean absolute deviation (MAD) is also reported with respect to (LNO-MBE-)CCSD(T), experiment and DMC.",
    replace_input= {},
    index = True,
    float_fmt = r"%.1f",
    filename = "Tables/SI_Table-X23_vdW_Bonded_Compare.tex",
    column_format="lrrrrrr"
)


,CCSD(T),CCSD(cT)-fit,DMC,DMC Error,Expt,Expt Error
"1,4-cyclohexanedione",-91.8,-89.9,-88.3,1.0,-87.5,6.6
Adamantane,-67.9,-65.8,-61.0,2.3,-64.9,4.7
Anthracene,-114.2,-108.9,-100.2,0.5,-104.2,7.5
Benzene,-52.8,-50.7,-49.8,0.2,-52.8,5.4
CO$_2$,-27.9,-27.4,-29.4,0.2,-29.1,1.0
Hexamine,-87.6,-85.5,-86.2,0.6,-83.7,4.4
Naphthalene,-87.1,-83.3,-75.5,0.5,-77.3,8.3
Pyrazine,-63.4,-61.1,-61.1,1.1,-63.2,4.4
Pyrazole,-78.9,-76.9,-77.3,0.5,-76.3,5.8
Triazine,-61.1,-59.4,-60.5,0.6,-62.2,4.3


### Figure 2

In [16]:
# Indices for each group
h_bonded_indices = [x23_idx for x23_idx in range(1, 24)
                    if x23_labels[x23_idx] in h_bonded_x23_labels]
vdw_bonded_indices = [x23_idx for x23_idx in range(1, 24)
                      if x23_labels[x23_idx] in vdw_bonded_x23_labels]
mixed_bonded_indices = [x23_idx for x23_idx in range(1, 24)
                        if x23_labels[x23_idx] not in h_bonded_x23_labels + vdw_bonded_x23_labels]
mixed_bonded_indices = [9, 22, 13]


# Number of rows in each panel
n_vdw   = len(vdw_bonded_indices)
n_mixed = len(mixed_bonded_indices)
n_hbond = len(h_bonded_indices)

# Figure and axes – height_ratios proportional to number of rows
fig, axs = plt.subplots(
    3, 1,
    figsize=(4.46, 8),
    constrained_layout=True,
    dpi=600,
    sharex=True,
    height_ratios=[n_vdw, n_mixed, n_hbond]
)

count = 0
system_dist = 1.5  # vertical spacing between systems (data units)

# --- Helper: set y-limits so that each row spans the same data range ---
def set_panel_ylim(ax, x_data, dist):
    """
    x_data: list of y-positions (0, dist, 2*dist, ...)
    dist: vertical spacing between rows in data units
    """
    ymin = x_data[0] - dist / 2
    ymax = x_data[-1] + dist / 2
    ax.set_ylim(ymax, ymin)  # inverted so top item appears at top


# ======================
# Panel 0: vdw-bonded
# ======================
x_data = []
counter = 0
left_y = []
left_label = []
right_y = []
right_label = []
for x23_idx in vdw_bonded_indices:
    y0 = counter * system_dist
    x_data.append(y0)

    system_exp_elatt_list = [
        float(x) - float(x23_exp_elatt_compile_dict[x23_labels[x23_idx]]['Etemp'])
        for x in x23_exp_elatt_compile_dict[x23_labels[x23_idx]]['Enthalpy measurements'].split(',')
    ]

    for elatt in system_exp_elatt_list:
        axs[0].plot(elatt, y0 - 0.3,
                    marker='o', markersize=3,
                    c='black', markeredgecolor='none', alpha=0.5)

    mean_exp = np.mean(system_exp_elatt_list)
    error = float(x23_exp_elatt_compile_dict[x23_labels[x23_idx]]['Error'])
    elatt_max = mean_exp + error
    elatt_min = mean_exp - error

    if count == 0:
        axs[0].barh(y0 - 0.3, elatt_max - elatt_min, 0.4, elatt_min,
                    color='grey', alpha=0.7, label='Exp. range')
        count += 1
    else:
        axs[0].barh(y0 - 0.3, elatt_max - elatt_min, 0.4, elatt_min,
                    color='grey', alpha=0.7)

    if counter % 2 == 0:
        left_y.append(y0)
        left_label.append(f'{x23_labels[x23_idx]}')
    else:
        right_y.append(y0)
        right_label.append(f'{x23_labels[x23_idx]}')    

    counter += 1

# DMC
axs[0].errorbar(
    [x23_dmc_elatt_dict[x23_idx][0] for x23_idx in vdw_bonded_indices],
    np.array(x_data) + 0.3,
    xerr=[x23_dmc_elatt_dict[x23_idx][1] for x23_idx in vdw_bonded_indices],
    marker='x', markersize=6, capsize=2,
    ls='none', label='DMC', c=color_dict['blue']
)

# CCSD(T)
axs[0].plot(
    [x23_final_elatt_dict[x23_idx]['CCSD(T)'] for x23_idx in vdw_bonded_indices],
    np.array(x_data) + 0.3,
    marker='x', markersize=6, ls='none',
    label='CCSD(T)', c=color_dict['red']
)

set_panel_ylim(axs[0], x_data, system_dist)
axs[0].tick_params(axis='y', which='both', labelright=True, right=True)

ax_left = axs[0]
ax_right = ax_left.twinx()

# Left side ticks + labels
ax_left.set_yticks(left_y)
ax_left.set_yticklabels(left_label)
ax_right.set_ylim(ax_left.get_ylim())

# Right side ticks + labels
ax_right.set_yticks(right_y)
ax_right.set_yticklabels(right_label)

# Left axis gridlines
ax_left.grid(axis='y', linestyle='--', color='grey', alpha=0.5)

# Make right axis gridlines also visible
ax_right.grid(axis='y', linestyle='--', color='grey', alpha=0.5)
ax_right.set_axisbelow(True)  # ensure right gridlines draw below data

# ======================
# Panel 1: mixed-bonded
# ======================
x_data = []
counter = 0
left_y = []
left_label = []
right_y = []
right_label = []
for x23_idx in mixed_bonded_indices:
    y0 = counter * system_dist
    x_data.append(y0)

    system_exp_elatt_list = [
        float(x) - float(x23_exp_elatt_compile_dict[x23_labels[x23_idx]]['Etemp'])
        for x in x23_exp_elatt_compile_dict[x23_labels[x23_idx]]['Enthalpy measurements'].split(',')
    ]

    for elatt in system_exp_elatt_list:
        axs[1].plot(elatt, y0 - 0.3,
                    marker='o', markersize=3,
                    c='black', markeredgecolor='none', alpha=0.5)

    mean_exp = np.mean(system_exp_elatt_list)
    error = float(x23_exp_elatt_compile_dict[x23_labels[x23_idx]]['Error'])
    elatt_max = mean_exp + error
    elatt_min = mean_exp - error

    if count == 0:
        axs[1].barh(y0 - 0.3, elatt_max - elatt_min, 0.4, elatt_min,
                    color='grey', alpha=0.7, label='Exp. range')
        count += 1
    else:
        axs[1].barh(y0 - 0.3, elatt_max - elatt_min, 0.4, elatt_min,
                    color='grey', alpha=0.7)

    if counter % 2 != 0:
        left_y.append(y0)
        left_label.append(f'{x23_labels[x23_idx]}')
    else:
        right_y.append(y0)
        right_label.append(f'{x23_labels[x23_idx]}')

    counter += 1

# DMC
axs[1].errorbar(
    [x23_dmc_elatt_dict[x23_idx][0] for x23_idx in mixed_bonded_indices],
    np.array(x_data) + 0.3,
    xerr=[x23_dmc_elatt_dict[x23_idx][1] for x23_idx in mixed_bonded_indices],
    marker='x', markersize=6, capsize=2,
    ls='none', label='DMC', c=color_dict['blue']
)

# CCSD(T)
axs[1].plot(
    [x23_final_elatt_dict[x23_idx]['CCSD(T)'] for x23_idx in mixed_bonded_indices],
    np.array(x_data) + 0.3,
    marker='x', markersize=6, ls='none',
    label='CCSD(T)', c=color_dict['red']
)

set_panel_ylim(axs[1], x_data, system_dist)
# axs[1].set_yticks(x_data, [f'{x23_idx:02d}' for x23_idx in mixed_bonded_indices])
ax_left = axs[1]
ax_right = ax_left.twinx()

# Left side ticks + labels
ax_left.set_yticks(left_y)
ax_left.set_yticklabels(left_label)
ax_right.set_ylim(ax_left.get_ylim())

# Right side ticks + labels
ax_right.set_yticks(right_y)
ax_right.set_yticklabels(right_label)

# Left axis gridlines
ax_left.grid(axis='y', linestyle='--', color='grey', alpha=0.5)

# Make right axis gridlines also visible
ax_right.grid(axis='y', linestyle='--', color='grey', alpha=0.5)
ax_right.set_axisbelow(True)  # ensure right gridlines draw below data

# ======================
# Panel 2: H-bonded
# ======================
x_data = []
counter = 0
left_y = []
left_label = []
right_y = []
right_label = []
for x23_idx in h_bonded_indices:
    y0 = counter * system_dist
    x_data.append(y0)

    system_exp_elatt_list = [
        float(x) - float(x23_exp_elatt_compile_dict[x23_labels[x23_idx]]['Etemp'])
        for x in x23_exp_elatt_compile_dict[x23_labels[x23_idx]]['Enthalpy measurements'].split(',')
    ]

    for elatt in system_exp_elatt_list:
        axs[2].plot(elatt, y0 - 0.3,
                    marker='o', markersize=3,
                    c='black', markeredgecolor='none', alpha=0.5)

    mean_exp = np.mean(system_exp_elatt_list)
    error = float(x23_exp_elatt_compile_dict[x23_labels[x23_idx]]['Error'])
    elatt_max = mean_exp + error
    elatt_min = mean_exp - error

    if count == 0:
        axs[2].barh(y0 - 0.3, elatt_max - elatt_min, 0.4, elatt_min,
                    color='grey', alpha=0.7, label='Exp. range')
        count += 1
    else:
        axs[2].barh(y0 - 0.3, elatt_max - elatt_min, 0.4, elatt_min,
                    color='grey', alpha=0.7)

    if counter % 2 == 0:
        left_y.append(y0)
        left_label.append(f'{x23_labels[x23_idx]}')
    else:
        right_y.append(y0)
        right_label.append(f'{x23_labels[x23_idx]}')

    counter += 1

# DMC
axs[2].errorbar(
    [x23_dmc_elatt_dict[x23_idx][0] for x23_idx in h_bonded_indices],
    np.array(x_data) + 0.3,
    xerr=[x23_dmc_elatt_dict[x23_idx][1] for x23_idx in h_bonded_indices],
    marker='x', markersize=6, capsize=2,
    ls='none', label='DMC', c=color_dict['blue']
)

# CCSD(T)
axs[2].plot(
    [x23_final_elatt_dict[x23_idx]['CCSD(T)'] for x23_idx in h_bonded_indices],
    np.array(x_data) + 0.3,
    marker='x', markersize=6, ls='none',
    label='CCSD(T)', c=color_dict['red']
)

set_panel_ylim(axs[2], x_data, system_dist)
axs[2].set_yticks(x_data, [f'{x23_idx:02d}' for x23_idx in h_bonded_indices])

axs[2].set_xlabel('Lattice energy (kJ/mol)')
axs[2].set_xlim([-200, 0])

ax_left = axs[2]
ax_right = ax_left.twinx()

# Left side ticks + labels
ax_left.set_yticks(left_y)
ax_left.set_yticklabels(left_label)
ax_right.set_ylim(ax_left.get_ylim())

# Right side ticks + labels
ax_right.set_yticks(right_y)
ax_right.set_yticklabels(right_label)

# Left axis gridlines
ax_left.grid(axis='y', linestyle='--', color='grey', alpha=0.5)

# Make right axis gridlines also visible
ax_right.grid(axis='y', linestyle='--', color='grey', alpha=0.5)
ax_right.set_axisbelow(True)  # ensure right gridlines draw below data

for label in axs[2].get_xticklabels():
    label.set_ha('left')   # horizontal alignment

# Slight extra padding between panels (doesn't change scale inside axes)
fig.set_constrained_layout_pads(h_pad=0.2)

plt.savefig('Figures/MAIN_Figure-X23_Comparison.png', dpi=600)

### Effect of (cT)-fit

In [17]:
fig, axs = plt.subplots(figsize=(5,7),constrained_layout=True,dpi=450)

count = 0

system_dist = 1.5
x_data=np.arange(23)*system_dist

for x23_idx in range(1,24):
    system_exp_elatt_list=[float(x) - float(x23_exp_elatt_compile_dict[x23_labels[x23_idx]]['Etemp']) for x in x23_exp_elatt_compile_dict[x23_labels[x23_idx]]['Enthalpy measurements'].split(',')]
    len_s=len(system_exp_elatt_list)
    for elatt in system_exp_elatt_list:
        axs.plot(elatt - float(x23_exp_elatt_compile_dict[x23_labels[x23_idx]]['Lattice energy']) ,(x23_idx-1)*system_dist-0.3,marker='o',markersize=3,c='black',markeredgecolor='none',alpha=0.5)

    mean_exp = float(x23_exp_elatt_compile_dict[x23_labels[x23_idx]]['Lattice energy'])
    error = float(x23_exp_elatt_compile_dict[x23_labels[x23_idx]]['Error'])
    elatt_max =  error
    elatt_min = - error
    if count == 0:
        axs.barh((x23_idx-1)*system_dist - 0.3, elatt_max-elatt_min,0.4,elatt_min,color='grey',alpha=0.7, label='Expt Error')
        count += 1
    else:
        axs.barh((x23_idx-1)*system_dist - 0.3, elatt_max-elatt_min,0.4,elatt_min,color='grey',alpha=0.7)

# Plot the DMC numbers
axs.errorbar([x23_dmc_elatt_dict[x23_idx][0] - float(x23_exp_elatt_compile_dict[x23_labels[x23_idx]]['Lattice energy']) for x23_idx in range(1,24)],x_data + 0.3,xerr=[x23_dmc_elatt_dict[x23_idx][1] for x23_idx in range(1,24)],marker='x',markersize=6,capsize=2,ls='none',label='DMC',c=color_dict['blue'])

# Plot the CC data
axs.plot([x23_final_elatt_dict[x23_idx]['CCSD(T)'] - float(x23_exp_elatt_compile_dict[x23_labels[x23_idx]]['Lattice energy']) for x23_idx in range(1,24)],x_data + 0.3,marker='x',markersize=6,ls='none',label='CCSD(T)',c=color_dict['red'])

# Plot the CCSD(cT) data
axs.plot([x23_final_elatt_dict[x23_idx]['CCSD(cT)-fit'] - float(x23_exp_elatt_compile_dict[x23_labels[x23_idx]]['Lattice energy']) for x23_idx in range(1,24)],x_data + 0.3,marker='x',markersize=6,ls='none',label='CCSD(cT)-fit',c=color_dict['green'])


axs.set_yticks(x_data,[x23_labels[x23_idx] for x23_idx in range(1,24)])

axs.legend(fontsize=8,loc='lower right')

axs.set_xlabel('Lattice energy relative to experiment (kJ/mol)')
axs.set_ylim([x_data[-1] + system_dist/2,x_data[0] - system_dist/2])
axs.set_xlim([-30,30])


axs.grid(color='grey', linestyle='-', linewidth=0.5,alpha=0.5)
plt.savefig('Figures/SI_Figure-cT_Effect.png')

## Effect of D4 dispersion subtractive embedding on distance cutoffs

In [18]:
cutoff_2b_dists = np.arange(3.5,12.5,0.5)
cutoff_3b_dists = np.arange(100,310,10)

x23_cutoff_2b_elatt_dict = {y: {f'{cutoff:.1f}': 0 for cutoff in cutoff_2b_dists} for y in list(range(1,24))}
x23_cutoff_3b_elatt_dict = {y: {cutoff: 0 for cutoff in cutoff_3b_dists} for y in list(range(1,24))}

x23_cutoff_2b_d4_elatt_dict = {y: {f'{cutoff:.1f}': 0 for cutoff in cutoff_2b_dists} for y in list(range(1,24))}
x23_cutoff_3b_d4_elatt_dict = {y: {cutoff: 0 for cutoff in cutoff_3b_dists} for y in list(range(1,24))}

for x23_idx in range(1,24):
    # Get the 2b data
    with connect(f'Data/X23/cWFT/x23_{x23_idx:02d}_2b.db') as db2:
        elatt_cwft = 0
        elatt_d4_cwft = 0
        dist_reached_bool = {cutoff: False for cutoff in cutoff_2b_dists}

        for i in range(len(db2)):
            row = db2.get(id=i+1)
            dist = row.data['distance']
            for cutoff in cutoff_2b_dists:
                if dist > cutoff and dist_reached_bool[cutoff] == False:
                    dist_reached_bool[cutoff] = True
                    x23_cutoff_2b_elatt_dict[x23_idx][f'{cutoff:.1f}'] = elatt_cwft/kjmol
                    x23_cutoff_2b_d4_elatt_dict[x23_idx][f'{cutoff:.1f}'] = elatt_d4_cwft/kjmol

            elatt_cwft += (row.data[f'CCSD(T) LAF TZ'] - row.data[f'MP2 LAF TZ'] + row.data[f'MP2 Canonical CBS'])*(row.data['count'])
            elatt_d4_cwft += (row.data[f'CCSD(T) LAF TZ'] - row.data[f'MP2 LAF TZ'] + row.data[f'MP2 Canonical CBS'] - row.data['D4'])*(row.data['count'])

        for cutoff in cutoff_2b_dists:
            if dist_reached_bool[cutoff] == False:
                x23_cutoff_2b_elatt_dict[x23_idx][f'{cutoff:.1f}'] = elatt_cwft/kjmol
                x23_cutoff_2b_d4_elatt_dict[x23_idx][f'{cutoff:.1f}'] = elatt_d4_cwft/kjmol
    # Get the 3b data
    with connect(f'Data/X23/cWFT/x23_{x23_idx:02d}_3b.db') as db3:
        elatt_cwft = 0
        elatt_d4_cwft = 0
        dist_reached_bool = {cutoff: False for cutoff in cutoff_3b_dists}

        for i in range(len(db3)):
            row = db3.get(id=i+1)
            dist = row.data['distance']
            for cutoff in cutoff_3b_dists:
                if dist > cutoff and dist_reached_bool[cutoff] == False:
                    dist_reached_bool[cutoff] = True
                    x23_cutoff_3b_elatt_dict[x23_idx][cutoff] = elatt_cwft/kjmol
                    x23_cutoff_3b_d4_elatt_dict[x23_idx][cutoff] = elatt_d4_cwft/kjmol

            elatt_cwft += (row.data[f'CCSD(T) LAF DZ'] - row.data[f'MP2 LAF DZ'] + row.data[f'MP2 Canonical CBS'])*(row.data['count'])
            elatt_d4_cwft += (row.data[f'CCSD(T) LAF DZ'] - row.data[f'MP2 LAF DZ'] + row.data[f'MP2 Canonical CBS'] - row.data['D4'])*(row.data['count'])

        for cutoff in cutoff_3b_dists:
            if dist_reached_bool[cutoff] == False:
                x23_cutoff_3b_elatt_dict[x23_idx][cutoff] = elatt_cwft/kjmol
                x23_cutoff_3b_d4_elatt_dict[x23_idx][cutoff] = elatt_d4_cwft/kjmol

    if replot_analysis:
        fig, axs = plt.subplots(1,2,figsize=(6.69,2.5),constrained_layout=True,dpi=300,sharey=True)
        
        # Plot the 2B convergence
        axs[0].plot(cutoff_2b_dists,np.array([x23_cutoff_2b_elatt_dict[x23_idx][f'{cutoff:.1f}'] for cutoff in cutoff_2b_dists]) - x23_cutoff_2b_elatt_dict[x23_idx]['12.0'],label=r'CCSD(T) [$E_\text{int}^\text{2b}=$' + f"{x23_cutoff_2b_elatt_dict[x23_idx]['12.0']:.1f} kJ/mol]",c=color_dict['red'],alpha=0.5,marker='x')

        axs[0].plot(cutoff_2b_dists,np.array([x23_cutoff_2b_d4_elatt_dict[x23_idx][f'{cutoff:.1f}'] for cutoff in cutoff_2b_dists]) - x23_cutoff_2b_d4_elatt_dict[x23_idx]['12.0'],ls='--',label=r'$\Delta^\text{CCSD(T)}_\text{D4}$ [$E_\text{int}^\text{2b}=$' + f"{x23_cutoff_2b_d4_elatt_dict[x23_idx]['12.0']:.1f} kJ/mol]",c=color_dict['blue'],alpha=0.7,marker='x')

        # Plot the 3B convergence
        axs[1].plot(cutoff_3b_dists,np.array([x23_cutoff_3b_elatt_dict[x23_idx][cutoff] for cutoff in cutoff_3b_dists]) - x23_cutoff_3b_elatt_dict[x23_idx][300],label=r'CCSD(T) [$E_\text{int}^\text{3b}=$' + f"{x23_cutoff_3b_elatt_dict[x23_idx][300]:.1f} kJ/mol]",c=color_dict['red'],alpha=0.5,marker='x')

        axs[1].plot(cutoff_3b_dists,np.array([x23_cutoff_3b_d4_elatt_dict[x23_idx][cutoff] for cutoff in cutoff_3b_dists]) - x23_cutoff_3b_d4_elatt_dict[x23_idx][300],ls='--',label=r'$\Delta^\text{CCSD(T)}_\text{D4}$ [$E_\text{int}^\text{3b}=$' + f"{x23_cutoff_3b_d4_elatt_dict[x23_idx][300]:.1f} kJ/mol]",c=color_dict['blue'],alpha=0.7,marker='x')

        # Fill between -1 and 1 kJ/mol
        axs[0].fill_between(cutoff_2b_dists,-1,1,color='grey',alpha=0.3,edgecolor='none')
        axs[1].fill_between(cutoff_3b_dists,-1,1,color='grey',alpha=0.3,edgecolor='none')


        axs[0].set_ylim([-4,4])
        axs[0].set_xlim([3.5,12])
        axs[1].set_xlim([100,300])

        axs[0].set_ylabel('Energy difference (kJ/mol)')
        axs[0].set_xlabel(r'2B Distance cutoff (Å)')
        axs[1].set_xlabel(r'3B Distance cutoff (Å$^3$)')

        axs[0].legend(fontsize=8)
        axs[1].legend(fontsize=8)

        # Make a title over both plots
        fig.suptitle(f'{x23_labels[x23_idx]}',fontsize=10)

        plt.savefig(f'Figures/SI_Figure-2b_3b_distance_convergence_{x23_idx:02d}.png')

# # Make the latex input str for including these tables

# latex_input_str = ""
# for x23_idx in range(1,24):
#     if x23_idx == 1:
#         latex_input_str += r"""\begin{figure}[h!]
# \includegraphics[width=\textwidth]{"""+ f"Figures/SI_Figure-2b_3b_distance_convergence_{x23_idx:02d}.png" + r"""}
# \caption{\label{fig:""" + f"x23_d4_conv_{x23_idx:02d}" + r"""} The convergence with distance cutoff of the two-body (2B) and three-body (3B) components of the many-body expansion (MBE) for the lattice energy $E_\text{latt}$ of """ + f"{x23_labels[x23_idx].lower()}" + r""" from the X23 dataset. The red line shows the CCSD(T) result, while the blue line shows the difference to the D4 result. The shaded area indicates the range of energies between -1 and 1 kJ/mol. We use the pairwise distance $R_{12}$ between the two monomers in the 2B case, and the product of the pairwise distances $R_{12}*R_{13}*R_{23}$ in the 3B case.}
# \end{figure}
# """
#     else:
#         latex_input_str += r"""\begin{figure}[h!]
# \includegraphics[width=\textwidth]{"""+ f"Figures/SI_Figure-2b_3b_distance_convergence_{x23_idx:02d}.png" + r"""}
# \caption{\label{fig:""" + f"x23_d4_conv_{x23_idx:02d}" + r"""} The convergence of the 2B and 3B MBE components to the $E_\text{latt}$ of """ + f"{x23_labels[x23_idx].lower()}" + r""".}
# \end{figure}
# """

# print(latex_input_str)

In [19]:
# Find the number of terms which are within 1 kJ/mol of the final value for each system
print("CCSD(T) correlation # of X23 systems within 1 kJ/mol of final 2B value at 9Å:", len([idx for idx in range(1,24) if abs(x23_cutoff_2b_elatt_dict[idx]['9.0'] - x23_cutoff_2b_elatt_dict[idx]['12.0']) < 1 ]))
print("Delta^{CCSD(T)}_{D4} # of X23 systems within 1 kJ/mol of final 2B value at 9Å:", len([idx for idx in range(1,24) if abs(x23_cutoff_2b_d4_elatt_dict[idx]['9.0'] - x23_cutoff_2b_d4_elatt_dict[idx]['12.0']) < 1 ]))
print("CCSD(T) correlation # of X23 systems within 1 kJ/mol of final 3B value at 200 Å³:", len([idx for idx in range(1,24) if abs(x23_cutoff_3b_elatt_dict[idx][200] - x23_cutoff_3b_elatt_dict[idx][300]) < 1 ]))
print("Delta^{CCSD(T)}_{D4} # of X23 systems within 1 kJ/mol of final 3B value at 200 Å³:", len([idx for idx in range(1,24) if abs(x23_cutoff_3b_d4_elatt_dict[idx][200] - x23_cutoff_3b_d4_elatt_dict[idx][300]) < 1 ]))

print("CCSD(T) correlation minimum 2B contribution:", np.min([x23_cutoff_2b_elatt_dict[idx]['12.0'] for idx in range(1,24)]))
print("CCSD(T) correlation maximum 2B contribution:", np.max([x23_cutoff_2b_elatt_dict[idx]['12.0'] for idx in range(1,24)]))
print("Delta^{CCSD(T)}_{D4} minimum 2B contribution:", np.min([x23_cutoff_2b_d4_elatt_dict[idx]['12.0'] for idx in range(1,24)]))
print("Delta^{CCSD(T)}_{D4} maximum 2B contribution:", np.max([x23_cutoff_2b_d4_elatt_dict[idx]['12.0'] for idx in range(1,24)]))

print("CCSD(T) correlation minimum 3B contribution:", np.min([x23_cutoff_3b_elatt_dict[idx][300] for idx in range(1,24)]))
print("CCSD(T) correlation maximum 3B contribution:", np.max([x23_cutoff_3b_elatt_dict[idx][300] for idx in range(1,24)]))
print("Delta^{CCSD(T)}_{D4} minimum 3B contribution:", np.min([x23_cutoff_3b_d4_elatt_dict[idx][300] for idx in range(1,24)]))
print("Delta^{CCSD(T)}_{D4} maximum 3B contribution:", np.max([x23_cutoff_3b_d4_elatt_dict[idx][300] for idx in range(1,24)]))

CCSD(T) correlation # of X23 systems within 1 kJ/mol of final 2B value at 9Å: 4
Delta^{CCSD(T)}_{D4} # of X23 systems within 1 kJ/mol of final 2B value at 9Å: 18
CCSD(T) correlation # of X23 systems within 1 kJ/mol of final 3B value at 200 Å³: 18
Delta^{CCSD(T)}_{D4} # of X23 systems within 1 kJ/mol of final 3B value at 200 Å³: 20
CCSD(T) correlation minimum 2B contribution: -171.67571689968193
CCSD(T) correlation maximum 2B contribution: -26.896983912428144
Delta^{CCSD(T)}_{D4} minimum 2B contribution: -5.606987673397395
Delta^{CCSD(T)}_{D4} maximum 2B contribution: 12.34793650773434
CCSD(T) correlation minimum 3B contribution: 0.2782924224286063
CCSD(T) correlation maximum 3B contribution: 7.157543899181965
Delta^{CCSD(T)}_{D4} minimum 3B contribution: -2.6038761018343437
Delta^{CCSD(T)}_{D4} maximum 3B contribution: 2.7680081857870764


### Smaller cutoff approaches

In [20]:
# Create table for the lattice energies with small, medium and large cutoffs
# Small: 2B cutoff 6.5A, 3B cutoff 100A^3
# Medium: 2B cutoff 9.0A, 3B cutoff 200A^3
# Large: 2B cutoff 12.0A, 3B cutoff 300A^3
x23_d4_cutoff_total_elatt_dict = {y: {'Small': 0, 'Medium': 0, 'Large': 0} for y in list(range(1,25))}


small_mad_list = []
medium_mad_list = []
large_mad_list = []

for x23_idx in range(1,24):
    small_elatt = x23_hf_d4_elatt_dict[x23_idx]/kjmol + x23_final_corr_d4_elatt_dict[x23_idx]['CCSD(T)']['1B']/kjmol + x23_cutoff_3b_d4_elatt_dict[x23_idx][100] + x23_cutoff_2b_d4_elatt_dict[x23_idx]['6.5']
    medium_elatt = x23_hf_d4_elatt_dict[x23_idx]/kjmol + x23_final_corr_d4_elatt_dict[x23_idx]['CCSD(T)']['1B']/kjmol+ x23_cutoff_3b_d4_elatt_dict[x23_idx][200] + x23_cutoff_2b_d4_elatt_dict[x23_idx]['9.0']
    large_elatt = x23_hf_d4_elatt_dict[x23_idx]/kjmol + x23_final_corr_d4_elatt_dict[x23_idx]['CCSD(T)']['1B']/kjmol + x23_cutoff_3b_d4_elatt_dict[x23_idx][300] + x23_cutoff_2b_d4_elatt_dict[x23_idx]['12.0']
    small_mad_list += [np.abs(small_elatt - large_elatt)]
    medium_mad_list += [np.abs(medium_elatt - large_elatt)]
    large_mad_list += [0]
    x23_d4_cutoff_total_elatt_dict[x23_idx]['Small'] = small_elatt
    x23_d4_cutoff_total_elatt_dict[x23_idx]['Medium'] = medium_elatt
    x23_d4_cutoff_total_elatt_dict[x23_idx]['Large'] = large_elatt

x23_d4_cutoff_total_elatt_dict[24] = {'Small': np.mean(small_mad_list), 'Medium': np.mean(medium_mad_list), 'Large': np.mean(large_mad_list)}

x23_d4_cutoff_total_elatt_df = pd.DataFrame.from_dict(x23_d4_cutoff_total_elatt_dict,orient='index')
x23_d4_cutoff_total_elatt_df.index = [f"{x23_labels[idx]}" if idx < 24 else 'MAD' for idx in x23_d4_cutoff_total_elatt_df.index]
x23_d4_cutoff_total_elatt_df.columns = ['Small','Medium','Large']
# Round to 1 decimal place
x23_d4_cutoff_total_elatt_df = x23_d4_cutoff_total_elatt_df.round(1)
display(x23_d4_cutoff_total_elatt_df)
# # Convert to latex
convert_df_to_latex_input(
    df = x23_d4_cutoff_total_elatt_df,
    start_input = '\\begin{table}[h]\n',
    label = "tab:x23_smaller_models",
    caption = r"Lattice energy (in kJ/mol) for the X23 dataset with the large (2B cutoff of $12.0\,$\AA{} and 3B cutoff of $300\,$\AA{}$^3$) LNO-MBE-CCSD(T) model, and the medium (2B cutoff of $6.5\,$\AA{} and 3B cutoff of $100\,$\AA{}$^3$) as well as small models (2B cutoff of $9.0\,$\AA{} and 3B cutoff of $200\,$\AA{}$^3$).",
    replace_input= {
        "[t]": "[c]"
    },
    index = True,
    filename = "Tables/SI_Table-X23_Smaller_CC_models.tex",
    float_fmt = r"%.1f",)

,Small,Medium,Large
"1,4-cyclohexanedione",-92.9,-92.2,-92.2
Acetic Acid,-70.9,-69.5,-70.3
Adamantane,-65.4,-66.9,-67.5
Ammonia,-37.4,-37.9,-38.6
Anthracene,-110.4,-111.3,-113.9
Benzene,-53.2,-52.3,-52.8
CO$_2$,-28.4,-28.3,-27.9
Cyanamide,-82.9,-81.8,-82.4
Cytosine,-163.8,-157.5,-161.9
Ethyl carbamate,-83.7,-85.1,-85.0


## Removing symmetrically equivalent subsystems with Coulomb matrix filter

In [21]:
# Validate that the filtering works

filtered_x23_final_corr_elatt_dict = {y:{x:{} for x in ['CCSD(T)']} for y in list(range(1,24))}

for x23_idx in range(1,24):
    system_all_methods_elatt_dict = {x:{} for x in ['MP2 LAF', 'CCSD LAF', 'CCSD(T) LAF', 'MP2 Canonical', 'MP2 Final', 'CCSD Final', 'CCSD(T) Final', 'CCSD(cT)-fit Final','D4']}
    system_total_time = 0
    method_elatt_dict = {'1B': 0, '2B': 0, '3B': 0, 'Total': 0}
    with connect(f'Data/X23/cWFT/x23_{x23_idx:02d}_1b.db') as db1:
        monomer_ene_list = []
        for i in range(2):
            row = db1.get(id=i+1)
            monomer_ene_list += [row.data[f'MP2 Canonical CBS'] - row.data[f'MP2 LAF QZ'] + row.data[f'CCSD(T) LAF QZ']]
        method_elatt_dict['1B'] = np.mean(monomer_ene_list[1:]) - monomer_ene_list[0]

    with connect(f'Data/X23/cWFT/filtered_x23_{x23_idx:02d}_2b.db') as db2:
        elatt_cwft = 0
        for i in range(len(db2)):
            row = db2.get(id=i+1)
            elatt_cwft += (row.data[f'CCSD(T) LAF TZ'] - row.data[f'MP2 LAF TZ'] + row.data[f'MP2 Canonical CBS'])*(row.data['count'])
        method_elatt_dict['2B'] = elatt_cwft

    with connect(f'Data/X23/cWFT/filtered_x23_{x23_idx:02d}_3b.db') as db3:
        elatt_cwft = 0
        for i in range(len(db3)):
            row = db3.get(id=i+1)
            elatt_cwft += (row.data[f'CCSD(T) LAF DZ'] - row.data[f'MP2 LAF DZ'] + row.data[f'MP2 Canonical CBS'])*(row.data['count'])
        method_elatt_dict['3B'] = elatt_cwft
    method_elatt_dict['Total'] = method_elatt_dict['1B'] + method_elatt_dict['2B'] + method_elatt_dict['3B']
    filtered_x23_final_corr_elatt_dict[x23_idx]['CCSD(T)'] = method_elatt_dict.copy()

## Computational cost analysis

### Analysis of distance cutoff dependence of cWFT MBE

In [22]:
cutoff_2b_dists = np.arange(3.5,12.5,0.5)
cutoff_3b_dists = np.arange(100,310,10)

# Cumulative cost in cpuh for each cutoff
x23_cutoff_2b_cpuh_dict = {y: {f'{cutoff:.1f}': 0 for cutoff in cutoff_2b_dists} for y in list(range(1,24))}
x23_cutoff_3b_cpuh_dict = {y: {cutoff: 0 for cutoff in cutoff_3b_dists} for y in list(range(1,24))}

# Collect the number of calculations for each cutoff
x23_cutoff_2b_num_calcs_dict = {y: {f'{cutoff:.1f}': 0 for cutoff in cutoff_2b_dists} for y in list(range(1,24))}
x23_cutoff_3b_num_calcs_dict = {y: {cutoff: 0 for cutoff in cutoff_3b_dists} for y in list(range(1,24))}

# Make filtered dictionaries
x23_cutoff_2b_cpuh_dict_filtered = {y: {f'{cutoff:.1f}': 0 for cutoff in cutoff_2b_dists} for y in list(range(1,24))}
x23_cutoff_3b_cpuh_dict_filtered = {y: {cutoff: 0 for cutoff in cutoff_3b_dists} for y in list(range(1,24))}
x23_cutoff_2b_num_calcs_dict_filtered = {y: {f'{cutoff:.1f}': 0 for cutoff in cutoff_2b_dists} for y in list(range(1,24))}
x23_cutoff_3b_num_calcs_dict_filtered = {y: {cutoff: 0 for cutoff in cutoff_3b_dists} for y in list(range(1,24))}

# Make full_enumeration dictionaries
x23_cutoff_2b_num_calcs_dict_full = {y: {f'{cutoff:.1f}': 0 for cutoff in cutoff_2b_dists} for y in list(range(1,24))}
x23_cutoff_3b_num_calcs_dict_full = {y: {cutoff: 0 for cutoff in cutoff_3b_dists} for y in list(range(1,24))}

for x23_idx in range(1,24):
    for mbe_type in ['filtered_','']:
        # Get the 2b data
        with connect(f'Data/X23/cWFT/{mbe_type}x23_{x23_idx:02d}_2b.db') as db2:
            cpuh_cwft = 0
            num_calcs_cwft = 0
            dist_reached_bool = {cutoff: False for cutoff in cutoff_2b_dists}

            for i in range(len(db2)):
                row = db2.get(id=i+1)
                dist = row.data['distance']
                for cutoff in cutoff_2b_dists:
                    if dist > cutoff and dist_reached_bool[cutoff] == False:
                        dist_reached_bool[cutoff] = True
                        x23_cutoff_2b_cpuh_dict[x23_idx][f'{cutoff:.1f}'] = cpuh_cwft
                        x23_cutoff_2b_num_calcs_dict[x23_idx][f'{cutoff:.1f}'] = num_calcs_cwft

                cpuh_cwft += row.data['time']*8/3600
                num_calcs_cwft += 1

            for cutoff in cutoff_2b_dists:
                if dist_reached_bool[cutoff] == False:
                    x23_cutoff_2b_cpuh_dict[x23_idx][f'{cutoff:.1f}'] = cpuh_cwft
                    x23_cutoff_2b_num_calcs_dict[x23_idx][f'{cutoff:.1f}'] = num_calcs_cwft
        # Get the 3b data
        with connect(f'Data/X23/cWFT/{mbe_type}x23_{x23_idx:02d}_3b.db') as db3:
            cpuh_cwft = 0
            num_calcs_cwft = 0
            dist_reached_bool = {cutoff: False for cutoff in cutoff_3b_dists}

            for i in range(len(db3)):
                row = db3.get(id=i+1)
                dist = row.data['distance']
                for cutoff in cutoff_3b_dists:
                    if dist > cutoff and dist_reached_bool[cutoff] == False:
                        dist_reached_bool[cutoff] = True
                        x23_cutoff_3b_cpuh_dict[x23_idx][cutoff] = cpuh_cwft
                        x23_cutoff_3b_num_calcs_dict[x23_idx][cutoff] = num_calcs_cwft

                cpuh_cwft += row.data['time']*8/3600
                num_calcs_cwft += 1

            for cutoff in cutoff_3b_dists:
                if dist_reached_bool[cutoff] == False:
                    x23_cutoff_3b_cpuh_dict[x23_idx][cutoff] = cpuh_cwft
                    x23_cutoff_3b_num_calcs_dict[x23_idx][cutoff] = num_calcs_cwft
        if mbe_type == 'filtered_':
            x23_cutoff_2b_cpuh_dict_filtered[x23_idx] = x23_cutoff_2b_cpuh_dict[x23_idx].copy()
            x23_cutoff_3b_cpuh_dict_filtered[x23_idx] = x23_cutoff_3b_cpuh_dict[x23_idx].copy()
            x23_cutoff_2b_num_calcs_dict_filtered[x23_idx] = x23_cutoff_2b_num_calcs_dict[x23_idx].copy()
            x23_cutoff_3b_num_calcs_dict_filtered[x23_idx] = x23_cutoff_3b_num_calcs_dict[x23_idx].copy()
    # Do full enumeration number of calculations (this is in PRL format)
    with connect(f'Data/X23/cWFT/full_enumeration/full_x23_{prl_to_x23_order[x23_idx]:02d}_2b.db') as db2:
        num_calcs_cwft = 0
        dist_reached_bool = {cutoff: False for cutoff in cutoff_2b_dists}

        for i in range(len(db2)):
            row = db2.get(id=i+1)
            dist = row.data['dist']
            for cutoff in cutoff_2b_dists:
                if dist > cutoff and dist_reached_bool[cutoff] == False:
                    dist_reached_bool[cutoff] = True
                    x23_cutoff_2b_num_calcs_dict_full[x23_idx][f'{cutoff:.1f}'] = num_calcs_cwft

            num_calcs_cwft += 1

        for cutoff in cutoff_2b_dists:
            if dist_reached_bool[cutoff] == False:
                x23_cutoff_2b_num_calcs_dict_full[x23_idx][f'{cutoff:.1f}'] = num_calcs_cwft
    # Get the 3b data
    with connect(f'Data/X23/cWFT/full_enumeration/full_x23_{prl_to_x23_order[x23_idx]:02d}_3b.db') as db3:
        num_calcs_cwft = 0
        dist_reached_bool = {cutoff: False for cutoff in cutoff_3b_dists}

        for i in range(len(db3)):
            row = db3.get(id=i+1)
            dist = row.data['dist']
            for cutoff in cutoff_3b_dists:
                if dist > cutoff and dist_reached_bool[cutoff] == False:
                    dist_reached_bool[cutoff] = True
                    x23_cutoff_3b_num_calcs_dict_full[x23_idx][cutoff] = num_calcs_cwft

            num_calcs_cwft += 1

        for cutoff in cutoff_3b_dists:
            if dist_reached_bool[cutoff] == False:
                x23_cutoff_3b_num_calcs_dict_full[x23_idx][cutoff] = num_calcs_cwft

#### 2B terms

In [23]:
# Convert cutoff dictionaries to table
cutoff_2b_dists = np.arange(5,13,1)
cutoff_2b_num_terms_df_dict = {}
for x23_idx in range(1, 24):
    cutoff_2b_num_terms_df_dict[(f'{x23_labels[x23_idx]}','Full Enumeration')] = {f'{cutoff:.1f}': x23_cutoff_2b_num_calcs_dict_full[x23_idx][f'{cutoff:.1f}'] for cutoff in cutoff_2b_dists}
    cutoff_2b_num_terms_df_dict[(f'{x23_labels[x23_idx]}','pMBE')] = {f'{cutoff:.1f}': x23_cutoff_2b_num_calcs_dict[x23_idx][f'{cutoff:.1f}'] for cutoff in cutoff_2b_dists}
    cutoff_2b_num_terms_df_dict[(f'{x23_labels[x23_idx]}','pMBE + Coulomb')] = {f'{cutoff:.1f}': x23_cutoff_2b_num_calcs_dict_filtered[x23_idx][f'{cutoff:.1f}'] for cutoff in cutoff_2b_dists}

cutoff_2b_table = pd.DataFrame(cutoff_2b_num_terms_df_dict).T
display(cutoff_2b_table)
# Convert to latex table
latex_input_str = '\n'.join(convert_df_to_latex_input(
df = cutoff_2b_table,
start_input = '\\begin{table}',
label = "tab:x23_2b_num_terms",
caption = "The number of two-body (2B) terms as a function of distance between the center of mass of the two molecules that make up the dimer subsystem.",
replace_input= {
    "[t]": "[c]"
},
output_str = True,
index = True).splitlines()[6:-4]) + '\n'

with open('Tables/SI_Table-pMBE_Coulomb_2B_Terms.tex', 'w') as f:
    f.write(r"""\LTcapwidth=\textwidth
    
\begin{longtable}{lrrrrrrrrrr}
\caption{\label{tab:x23_2b_num_terms}The number of two-body (2B) terms as a function of distance between the center of mass of the two molecules that make up the dimer subsystem.} \\

\toprule
2B distance (\AA) &  & 5 & 6 & 7 & 8 & 9 & 10 & 11 & 12 \\
\midrule
\endfirsthead



\caption[]{(continued)} \\
\endhead

\multicolumn{11}{r}{{Continued on next page}} \\
\endfoot

\bottomrule
\endlastfoot

""")
    f.write(latex_input_str)
    f.write(r"\end{longtable}")

5.0  6.0  7.0  8.0  9.0  10.0  11.0  \
1,4-cyclohexanedione Full Enumeration    2    6   14   14   16    26    46   
                     pMBE                2    6   11   11   12    18    35   
                     pMBE + Coulomb      1    3    7    7    8    13    23   
Acetic Acid          Full Enumeration    6   12   22   30   36    62    76   
                     pMBE                5   10   18   25   31    55    69   
...                                    ...  ...  ...  ...  ...   ...   ...   
Uracil               pMBE                1    3   12   21   23    28    42   
                     pMBE + Coulomb      1    3    8   15   16    19    30   
Urea                 Full Enumeration   10   14   14   30   42    60    76   
                     pMBE                9   11   11   21   33    46    58   
                     pMBE + Coulomb      3    4    4    8   10    14    16   

                                       12.0  
1,4-cyclohexanedione Full Enumeration    50  
                     pMBE                39  
                     pMBE + Coulomb      25  
Acetic Acid          Full Enumeration   100  
                     pMBE                91  
...                                     ...  
Uracil               pMBE                56  
                     pMBE + Coulomb      40  
Urea                 Full Enumeration    96  
                     pMBE                76  
                     pMBE + Coulomb      20  

[69 rows x 8 columns]

#### 3B terms

In [24]:
# Convert cutoff dictionaries to table
cutoff_3b_dists = np.arange(120,310,30)
cutoff_3b_num_terms_df_dict = {}
for x23_idx in range(1, 24):
    cutoff_3b_num_terms_df_dict[(f'{x23_labels[x23_idx]}','Full Enumeration')] = {cutoff: x23_cutoff_3b_num_calcs_dict_full[x23_idx][cutoff] for cutoff in cutoff_3b_dists}
    cutoff_3b_num_terms_df_dict[(f'{x23_labels[x23_idx]}','pMBE')] = {cutoff: x23_cutoff_3b_num_calcs_dict[x23_idx][cutoff] for cutoff in cutoff_3b_dists}
    cutoff_3b_num_terms_df_dict[(f'{x23_labels[x23_idx]}','pMBE + Coulomb')] = {cutoff: x23_cutoff_3b_num_calcs_dict_filtered[x23_idx][cutoff] for cutoff in cutoff_3b_dists}

cutoff_3b_table = pd.DataFrame(cutoff_3b_num_terms_df_dict).T
display(cutoff_3b_table)
# Convert to latex table
latex_input_str = '\n'.join(convert_df_to_latex_input(
df = cutoff_3b_table,
start_input = '\\begin{table}',
label = "tab:x23_3b_num_terms",
caption = "The number of three-body (3B) terms as a function of the product of the pairwise distances between the center of mass of the three molecules that make up the trimer subsystem.",
replace_input= {
    "[t]": "[c]"
},
output_str = True,
index = True).splitlines()[6:-4]) + '\n'

with open('Tables/SI_Table-pMBE_Coulomb_3B_Terms.tex', 'w') as f:
    f.write(r"""\LTcapwidth=\textwidth
    
\begin{longtable}{lrrrrrrrrr}
\caption{\label{tab:x23_3b_num_terms}The number of three-body (3B) terms as a function of the product of the pairwise distances between the center of mass of the three molecules that make up the trimer subsystem.} \\

\toprule
3B distance (\AA\textsuperscript{3}) &  & 120 & 150 & 180 & 210 & 240 & 270 & 300 \\
\midrule
\endfirsthead



\caption[]{(continued)} \\
\endhead

\multicolumn{10}{r}{{Continued on next page}} \\
\endfoot

\bottomrule
\endlastfoot

""")
    f.write(latex_input_str)
    f.write(r"\end{longtable}")



120  150  180  210  240  270  300
1,4-cyclohexanedione Full Enumeration    0    0    3   15   27   48   60
                     pMBE                0    0    2   10   18   32   40
                     pMBE + Coulomb      0    0    1    5    9   16   20
Acetic Acid          Full Enumeration   30   45   87  135  177  204  279
                     pMBE               20   31   59   97  131  150  205
...                                    ...  ...  ...  ...  ...  ...  ...
Uracil               pMBE                3    7   19   30   44   52   75
                     pMBE + Coulomb      2    4   16   23   31   35   50
Urea                 Full Enumeration   24   42   78  135  183  219  255
                     pMBE               16   28   52   81  113  133  157
                     pMBE + Coulomb      3    6   10   19   23   27   31

[69 rows x 7 columns]

# Analysis DMC costs

In [25]:
# Convert csv to dict for file "Data/X23/cWFT/DMC_Expt_References/DMCcost.csv"
dmc_cost_df = pd.read_csv("Data/X23/DMC/DMCcost.csv",index_col=0)
display(dmc_cost_df)
dmc_cost_dict = dmc_cost_df.to_dict(orient='index')

,Ne,Ne_mol,simcell,Nmol,Nscell,Nmolpcell,nome,DMC[4],DMC[2],DMC[1],DMC[0.5]
mol,,,,,,,,,,,
01CYHEXO,704,44,2x2x2,16.0,8,2.0,"1,4-cyclohexanedione",297,1984,23818,317584
02acetic_acid,576,24,1x3x2,24.0,6,4.0,Acetic Acid,64,433,5198,69307
03adamantane,896,56,2x2x2,16.0,8,2.0,Adamantane,650,4334,52013,693512
04ammonia,256,8,2x2x2,32.0,8,4.0,Ammonia,2,18,219,2925
05anthracene,264,66,1x2x1,4.0,2,2.0,Anthracene,309,2065,24791,330552
06benzene,960,30,2x2x2,32.0,8,4.0,Benzene,183,1221,14657,195439
07co2,512,16,2x2x2,32.0,8,4.0,CO2,24,160,1926,25689
08cyanamide,1024,16,2x2x2,64.0,8,8.0,Cyanamide,52,349,4198,55984
09cytosine,504,42,1x1x3,12.0,3,4.0,Cytosine,192,1285,15430,205738


### Overall comparison to periodic HF and DMC

In [28]:
# Compute the cost for periodic QE calculations

x23_method_cost_dict = {y: {'HF': 0 , 'DFT (GGA)': 0, 'CCSD(T) [low]': 0, 'CCSD(T) [moderate]': 0 , 'CCSD(T) [high]': 0, 'DMC [4]': 0 ,'DMC [2]': 0, 'DMC [1]': 0} for y in list(range(1,24))}

# QE data is in PRL order
for x23_idx in range(1,24):
    x23_method_cost_dict[x23_idx]['HF'] = get_qe_walltime(f'Data/X23/HF/QE_cost/hybrid/{prl_to_x23_order[x23_idx]:02d}/solid/pw.out.gz')*192 + get_qe_walltime(f'Data/X23/HF/QE_cost/hybrid/{prl_to_x23_order[x23_idx]:02d}/molecule/pw.out.gz')*192
    x23_method_cost_dict[x23_idx]['DFT (GGA)'] = get_qe_walltime(f'Data/X23/HF/QE_cost/GGA/{prl_to_x23_order[x23_idx]:02d}/solid/pw.out.gz')*192 + get_qe_walltime(f'Data/X23/HF/QE_cost/GGA/{prl_to_x23_order[x23_idx]:02d}/molecule/pw.out.gz')*192

    x23_method_cost_dict[x23_idx]['CCSD(T) [low]'] = x23_cutoff_2b_cpuh_dict_filtered[x23_idx]['6.5'] + x23_cutoff_3b_cpuh_dict_filtered[x23_idx][100]
    x23_method_cost_dict[x23_idx]['CCSD(T) [moderate]'] = x23_cutoff_2b_cpuh_dict_filtered[x23_idx]['9.0'] + x23_cutoff_3b_cpuh_dict_filtered[x23_idx][200]
    x23_method_cost_dict[x23_idx]['CCSD(T) [high]'] = x23_cutoff_2b_cpuh_dict_filtered[x23_idx]['12.0'] + x23_cutoff_3b_cpuh_dict_filtered[x23_idx][300]

dmc_dict_prl_labels = list(dmc_cost_dict.keys())

# DMC data is in PRL order and given 1sigma error bars, we want 2sigma error bars for 95% confidence interval
for x23_idx, x23_idx_label in enumerate(dmc_cost_dict.keys()):
    # Cost for 1 kJ/mol std
    x23_method_cost_dict[x23_idx+1]['DMC [1]'] = dmc_cost_dict[dmc_dict_prl_labels[prl_to_x23_order[x23_idx+1]-1]]['DMC[0.5]']
    # Cost for 2 kJ/mol std
    x23_method_cost_dict[x23_idx+1]['DMC [2]'] = dmc_cost_dict[dmc_dict_prl_labels[prl_to_x23_order[x23_idx+1]-1]]['DMC[1]']
    # Cost for 4 kJ/mol std
    x23_method_cost_dict[x23_idx+1]['DMC [4]'] = dmc_cost_dict[dmc_dict_prl_labels[prl_to_x23_order[x23_idx+1]-1]]['DMC[2]']

# Add a row for the mean
x23_method_cost_dict['Mean'] = {method: np.mean([x23_method_cost_dict[idx][method] for idx in range(1,24)]) for method in x23_method_cost_dict[1].keys()}

# Convert to dataframe
x23_method_cost_df = pd.DataFrame.from_dict(x23_method_cost_dict,orient='index')
x23_method_cost_df.index = [f"{x23_labels[idx]}" for idx in range(1,24)] + ['Mean']
# Round to integer
x23_method_cost_df = x23_method_cost_df.round(0).astype(int)

display(x23_method_cost_df)
# Convert to latex
convert_df_to_latex_input(
    df = x23_method_cost_df,
    start_input = '\\begin{table}[h]\n',
    label = "tab:x23_method_cost",
    caption = r"Estimated computational cost (in CPUh) for the different methods used to compute the lattice energies of the X23 dataset. The LNO-MBE-CCSD(T) costs are given for the low (2B cutoff of $6.5\,$\AA{} and 3B cutoff of $100\,$\AA{}$^3$), moderate (2B cutoff of $9.0\,$\AA{} and 3B cutoff of $200\,$\AA{}$^3$) and high (2B cutoff of $12.0\,$\AA{} and 3B cutoff of $300\,$\AA{}$^3$) models. The DMC costs are estimated for a standard deviation of 2$\sigma$ of 1, 2 and 4 kJ/mol, respectively.",
    replace_input= {
        "[t]": "[c]"
    },
    index = True,
    rotate_column_header=True,
    filename = "Tables/SI_Table-X23_Method_Costs.tex")

np.save('Data/X23/X23_Costs.npy', x23_method_cost_dict)

,HF,DFT (GGA),CCSD(T) [low],CCSD(T) [moderate],CCSD(T) [high],DMC [4],DMC [2],DMC [1]
"1,4-cyclohexanedione",1297,3,800,2276,7824,1984,23818,317584
Acetic Acid,1684,4,139,878,2195,433,5198,69307
Adamantane,365,3,624,1795,5585,4334,52013,693512
Ammonia,93,1,50,193,411,18,219,2925
Anthracene,2496,6,2822,6586,21941,2065,24791,330552
Benzene,1201,4,241,1000,2802,1221,14657,195439
CO$_2$,162,1,24,116,277,160,1926,25689
Cyanamide,1993,5,137,465,1149,349,4198,55984
Cytosine,2778,6,754,5566,10764,1285,15430,205738
Ethyl carbamate,1048,3,484,1604,4160,1044,12528,167046


## Convergence tests on Benzene

### DeltaMP2 vs LAF vs Canonical approach

In [30]:
# Calculating MP2, CCSD, CCSD(T) and CCSD(cT) results

for x23_idx in [6]:
    benzene_basis_elatt_dict = {x:{} for x in ['CCSD(T) tight','CCSD(T) vtight','CCSD(T) LAF', 'CCSD(T) Final']}
    for method in ['CCSD(T) tight','CCSD(T) vtight','CCSD(T) LAF', 'CCSD(T) Final']:
        method_elatt_dict = {'1B': 0, '2B': 0, '3B': 0, 'Total': 0}

        with connect(f'Data/X23/cWFT/x23_{x23_idx:02d}_1b.db') as db1_1:
            with connect(f'Data/cWFT_Convergence/benzene_1b_basis.db') as db1:
                monomer_ene_list = []
                for i in range(2):
                    row = db1.get(id=i+1)
                    row_1 = db1_1.get(id=i+1)
                    if method == 'CCSD(T) LAF':
                        monomer_ene_list += [row.data[f'CCSD(T) LAF CBS']]
                    elif method == 'CCSD(T) Final':
                        monomer_ene_list += [row.data[f'CCSD(T) Canonical QZ'] + row_1.data['MP2 Canonical CBS'] - row_1.data['MP2 Canonical QZ']]
                    elif method == 'CCSD(T) tight':
                        monomer_ene_list += [row.data[f'CCSD(T) tight CBS']]
                    elif method == 'CCSD(T) vtight':
                        monomer_ene_list += [row.data[f'CCSD(T) vtight CBS']]
                        
                method_elatt_dict['1B'] = np.mean(monomer_ene_list[1:]) - monomer_ene_list[0]

        with connect(f'Data/X23/cWFT/x23_{x23_idx:02d}_2b.db') as db2_1:
            with connect(f'Data/cWFT_Convergence/benzene_2b_basis.db') as db2:
                elatt_cwft = 0
                for i in range(len(db2)):
                    row = db2.get(id=i+1)
                    row_1 = db2_1.get(id=i+1)
                    if method == 'CCSD(T) LAF':
                        elatt_cwft += row.data[f'CCSD(T) LAF CBS']*(row.data['count'])
                    elif method == 'CCSD(T) Final':
                        elatt_cwft += (row.data[f'CCSD(T) Canonical TZ'] + row_1.data['MP2 Canonical CBS'] - row_1.data['MP2 Canonical TZ'])*(row.data['count'])
                    elif method == 'CCSD(T) tight':
                        elatt_cwft += row.data[f'CCSD(T) tight CBS']*(row.data['count'])
                    elif method == 'CCSD(T) vtight':
                        elatt_cwft += row.data[f'CCSD(T) vtight CBS']*(row.data['count'])

                method_elatt_dict['2B'] = elatt_cwft

        with connect(f'Data/X23/cWFT/x23_{x23_idx:02d}_3b.db') as db3_1:
            with connect(f'Data/cWFT_Convergence/benzene_3b_basis.db') as db3:
                elatt_cwft = 0
                for i in range(len(db3)):
                    row = db3.get(id=i+1)
                    row_1 = db3_1.get(id=i+1)
                    if method == 'CCSD(T) LAF':
                        elatt_cwft += row.data[f'CCSD(T) LAF CBS']*(row.data['count'])
                    elif method == 'CCSD(T) Final':
                        elatt_cwft += (row.data[f'CCSD(T) Canonical DZ'] + row_1.data['MP2 Canonical CBS'] - row_1.data['MP2 Canonical DZ'])*(row.data['count'])
                    elif method == 'CCSD(T) tight':
                        elatt_cwft += row.data[f'CCSD(T) tight CBS']*(row.data['count'])
                    elif method == 'CCSD(T) vtight':
                        elatt_cwft += row.data[f'CCSD(T) vtight CBS']*(row.data['count'])
                method_elatt_dict['3B'] = elatt_cwft
        method_elatt_dict['Total'] = method_elatt_dict['1B'] + method_elatt_dict['2B'] + method_elatt_dict['3B']
        benzene_basis_elatt_dict[method] = method_elatt_dict.copy()


In [31]:
# Benzene LNO convergence table

benzene_lno_conv_df = pd.DataFrame.from_dict(
    {
        'tight': {key: value / kjmol for key,value in benzene_basis_elatt_dict['CCSD(T) tight'].items()},
        'vtight': {key: value / kjmol for key,value in benzene_basis_elatt_dict['CCSD(T) vtight'].items()},
        'LAF': {key: value / kjmol for key,value in benzene_basis_elatt_dict['CCSD(T) LAF'].items()},
        'DeltaMP2': {key: value / kjmol for key,value in x23_final_corr_elatt_dict[6]['CCSD(T)'].items()},
        'Canonical': {key: value / kjmol for key,value in benzene_basis_elatt_dict['CCSD(T) Final'].items()}
    }, orient='index')

# Round to 1 d.p.
benzene_lno_conv_df = benzene_lno_conv_df.round(1)

display(benzene_lno_conv_df)

# Convert to LaTeX table
convert_df_to_latex_input(
    df = benzene_lno_conv_df,
    start_input = '\\begin{table}[h]\n',
    label = "tab:x23_benzene_lno_conv",
    caption = r"Convergence of the LNO-MBE-CCSD(T) lattice energy of benzene with respect to the LNO threshold, compared to the canonical result. All values are in kJ/mol.",
    replace_input= {'DeltaMP2':r'Composite',
        r'vtight ': r'{\tt vTight} ',
        r'tight ': r'{\tt Tight} ',
        r'normal ': r'{\tt Normal} ',
        r'loose ': r'{\tt Loose} '},
    float_fmt="%.1f",
    filename = "Tables/SI_Table-X23_Benzene_LNO_Approach_Convergence.tex")

,1B,2B,3B,Total
tight,-1.3,-80.0,3.6,-77.7
vtight,-1.3,-79.3,5.8,-74.8
LAF,-1.3,-78.9,6.8,-73.4
DeltaMP2,-1.3,-79.6,5.3,-75.7
Canonical,-1.3,-79.0,4.7,-75.6


### LNO thresholds

In [33]:
# Benzene LNO threshold convergence

benzene_conv_lno_dict = {(y,x): 0 for y in ['2B','3B'] for x in ['tight','vtight','LAF','Canonical'] }

with connect('Data/cWFT_Convergence/benzene_2b_lno_conv.db') as db2:
    for i in range(1,len(db2)+1):
        row = db2.get(id=i)
        benzene_conv_lno_dict[('2B','tight')] += row.data['LNO-CCSD(T) tight aug-cc-pVDZ']*row.data['count']/ (4*kjmol)
        benzene_conv_lno_dict[('2B','vtight')] += row.data['LNO-CCSD(T) vtight aug-cc-pVDZ']*row.data['count']/(4*kjmol)
        benzene_conv_lno_dict[('2B','LAF')] += (row.data['LNO-CCSD(T) vtight aug-cc-pVDZ'] + 0.5*(row.data['LNO-CCSD(T) vtight aug-cc-pVDZ'] - row.data['LNO-CCSD(T) tight aug-cc-pVDZ'])) *row.data['count']/(4*kjmol)
        benzene_conv_lno_dict[('2B','Canonical')] += row.data['DF-CCSD(T) aug-cc-pVDZ']*row.data['count']/(4*kjmol)

with connect('Data/cWFT_Convergence/benzene_3b_lno_conv.db') as db3:
    for i in range(1,len(db3)+1):
        row = db3.get(id=i)
        benzene_conv_lno_dict[('3B','tight')] += row.data['LNO-CCSD(T) tight aug-cc-pVDZ']*row.data['count']/(4*kjmol)
        benzene_conv_lno_dict[('3B','vtight')] += row.data['LNO-CCSD(T) vtight aug-cc-pVDZ']*row.data['count']/(4*kjmol)
        benzene_conv_lno_dict[('3B','LAF')] += (row.data['LNO-CCSD(T) vtight aug-cc-pVDZ'] + 0.5*(row.data['LNO-CCSD(T) vtight aug-cc-pVDZ'] - row.data['LNO-CCSD(T) tight aug-cc-pVDZ'])) *row.data['count']/(4*kjmol)
        benzene_conv_lno_dict[('3B','Canonical')] += row.data['DF-CCSD(T) aug-cc-pVDZ']*row.data['count']/(4*kjmol)

# Convert to DataFrame
benzene_conv_lno_df = pd.DataFrame(benzene_conv_lno_dict, index=['CCSD(T) aug-cc-pVDZ (kJ/mol)'])
# To nearest 1 d.p.
benzene_conv_lno_df = benzene_conv_lno_df.round(1)
# Write to latex table
display(benzene_conv_lno_df)
# Convert to LaTeX table
convert_df_to_latex_input(
    df = benzene_conv_lno_df,
    start_input = '\\begin{table}[h]\n',
    label = "tab:benzene_lno_conv",
    caption = r"Convergence of 2B and 3B contributions, for the benzene molecular crystal in the X23 dataset, with LNO thresholds: {\tt Tight}, {\tt vTight} and extrapolated locality approximation free (LAF) for aug-cc-pVDZ basis set",
    float_fmt="%.1f",
    replace_input={
        r'{3B} \\': r'{3B} \\ \cmidrule(lr){2-5} \cmidrule(lr){6-9}',
        r' vtight ': r' {\tt vTight} ',
        r' tight ': r' {\tt Tight} ',
        r' normal ': r' {\tt Normal} ',
        r' loose ': r' {\tt Loose} ',
    },
    filename = "Tables/SI_Table-Benzene_LNO_Conv_Effect.tex")

2B                           3B              \
                             tight vtight   LAF Canonical tight vtight  LAF   
CCSD(T) aug-cc-pVDZ (kJ/mol) -70.7  -70.2 -70.0     -70.0   4.7    4.6  4.5   

                                        
                             Canonical  
CCSD(T) aug-cc-pVDZ (kJ/mol)       5.1

### Basis set

In [34]:
# Benzene basis set convergence
benzene_basis_conv_dict = {x:0 for x in [('2B','TZ'), ('2B','QZ'), ('2B','CBS'), ('3B','DZ'), ('3B','TZ'), ('3B','CBS')]}
for x23_idx in [6]:
    with connect(f'Data/X23/cWFT/x23_{x23_idx:02d}_2b.db') as db2:
        for i in range(len(db2)):
            row = db2.get(id=i+1)
            benzene_basis_conv_dict[('2B','TZ')] += row.data['MP2 Canonical TZ']*(row.data['count'])/(4*kjmol)
            benzene_basis_conv_dict[('2B','QZ')] += row.data['MP2 Canonical QZ']*(row.data['count'])/(4*kjmol)
            benzene_basis_conv_dict[('2B','CBS')] += row.data['MP2 Canonical CBS']*(row.data['count'])/(4*kjmol)

    with connect(f'Data/X23/cWFT/x23_{x23_idx:02d}_3b.db') as db3:
        for i in range(len(db3)):
            row = db3.get(id=i+1)
            benzene_basis_conv_dict[('3B','DZ')] += row.data['MP2 Canonical DZ']*(row.data['count'])/(4*kjmol)
            benzene_basis_conv_dict[('3B','TZ')] += row.data['MP2 Canonical TZ']*(row.data['count'])/(4*kjmol)
            benzene_basis_conv_dict[('3B','CBS')] += row.data['MP2 Canonical CBS']*(row.data['count'])/(4*kjmol)

# Convert to DataFrame
benzene_basis_conv_df = pd.DataFrame(benzene_basis_conv_dict, index=['MP2 (kJ/mol)']).T
# To nearest 1 d.p.
benzene_basis_conv_df = benzene_basis_conv_df.round(1)
# Write to latex table
display(benzene_basis_conv_df)
# Convert to LaTeX table
convert_df_to_latex_input(
    df = benzene_basis_conv_df,
    start_input = '\\begin{table}[h]\n',
    label = "tab:benzene_basis_conv",
    caption = r"Convergence of 2B with basis set: aug-cc-pVTZ, aug-cc-pVQZ and CBS(aug-cc-pVTZ/aug-cc-pVQZ) as well as 3B with basis set: aug-cc-pVDZ, aug-cc-pVTZ and CBS(aug-cc-pVDZ/aug-cc-pVTZ) for the MP2 level of theory.",
    replace_input= {"lr":"lrr",r'[t]': r'[c]',},
    float_fmt="%.1f",
    filename = "Tables/SI_Table-Benzene_Basis_Conv_Effect.tex") 

MP2 (kJ/mol)
2B TZ          -22.5
   QZ          -22.9
   CBS         -23.2
3B DZ            0.5
   TZ            0.5
   CBS           0.5

# Lattice and relative energies of ICE13

## cWFT MBE analysis

In [35]:
# Calculating MP2, CCSD, CCSD(T) and CCSD(cT) results

ice13_systems = {
    1: 'Ih',
    2: 'II',
    3: 'III',
    4: 'IV',
    6: 'VI',
    7: 'VII',
    8: 'VIII',
    9: 'IX',
    11: 'XI',
    13: 'XIII',
    14: 'XIV',
    15: 'XV',
    17: 'XVII'
}

ice13_methods_dict = {
    'AFsupercell': ['MP2 LAF', 'CCSD(T) LAF'],
    'AFcell': ['MP2 Canonical', 'CCSD Canonical', 'CCSD(T) LAF','CCSD(T) Canonical', 'CCSD(T) DeltaMP2'],
    'Unitcell': ['MP2 Canonical','CCSD Canonical', 'CCSD(T) LAF','CCSD(T) Canonical', 'CCSD(T) DeltaMP2']
}

ice13_final_corr_elatt_dict = {
    'AFsupercell': {y:{x:{} for x in ['MP2 LAF', 'CCSD LAF', 'CCSD(T) LAF']} for y in [1,3,4,6,7,17]},
    'AFcell': {y:{x:{} for x in ['MP2 Canonical', 'CCSD Canonical' 'CCSD(T) LAF','CCSD(T) Canonical', 'CCSD(T) DeltaMP2']} for y in [1,3,4,7,17]},
    'Unitcell': {y:{x:{} for x in ['MP2 Canonical', 'CCSD Canonical', 'CCSD(T) LAF','CCSD(T) Canonical', 'CCSD(T) DeltaMP2']} for y in ice13_systems.keys()}
}

for cell_size in ['AFsupercell','AFcell','Unitcell']:
    for ice13_idx in ice13_final_corr_elatt_dict[cell_size].keys():
        system_all_methods_elatt_dict = {x:{} for x in ice13_methods_dict[cell_size]}
        for method in ice13_methods_dict[cell_size]:
            method_elatt_dict = {'1B': 0, '2B': 0, '3B': 0, 'Total': 0}
            with connect(f'Data/ICE13/cWFT/{cell_size}/ice13_{ice13_idx:02d}_1b.db') as db1:
                num_monomers = len(db1) - 1
                monomer_ene_list = []
                for i in range(len(db1)):
                    row = db1.get(id=i+1)
                    if 'DeltaMP2' in method:
                        monomer_ene_list += [row.data['CCSD(T) Canonical QZ'] - row.data['MP2 Canonical QZ'] + row.data['MP2 Canonical CBS']]
                    else:
                        monomer_ene_list += [row.data[f'{method} CBS']]
                method_elatt_dict['1B'] = np.mean(monomer_ene_list[1:]) - monomer_ene_list[0]

            with connect(f'Data/ICE13/cWFT/{cell_size}/ice13_{ice13_idx:02d}_2b.db') as db2:
                elatt_cwft = 0
                for i in range(len(db2)):
                    row = db2.get(id=i+1)
                    if 'DeltaMP2' in method:
                        elatt_cwft += (row.data['CCSD(T) Canonical TZ'] - row.data['MP2 Canonical TZ'] + row.data['MP2 Canonical CBS'])*(row.data['count'])
                    else:
                        elatt_cwft += row.data[f'{method} CBS']*(row.data['count'])
                method_elatt_dict['2B'] = elatt_cwft/num_monomers

            if ice13_idx == 7 and cell_size == 'AFsupercell':
                with connect(f'Data/ICE13/cWFT/{cell_size}/ice13_{ice13_idx:02d}_3b_01.db') as db3_1:
                    elatt_cwft = 0
                    for i in range(len(db3_1)):
                        row = db3_1.get(id=i+1)
                        if 'DeltaMP2' in method:
                            elatt_cwft += (row.data['CCSD(T) Canonical DZ'] - row.data['MP2 Canonical DZ'] + row.data['MP2 Canonical CBS'])*(row.data['count'])
                        else:
                            elatt_cwft += row.data[f'{method} CBS']*(row.data['count'])
                with connect(f'Data/ICE13/cWFT/{cell_size}/ice13_{ice13_idx:02d}_3b_02.db') as db3_2:
                    for i in range(len(db3_2)):
                        row = db3_2.get(id=i+1)
                        if 'DeltaMP2' in method:
                            elatt_cwft += (row.data['CCSD(T) Canonical DZ'] - row.data['MP2 Canonical DZ'] + row.data['MP2 Canonical CBS'])*(row.data['count'])
                        else:
                            elatt_cwft += row.data[f'{method} CBS']*(row.data['count'])
            else:
                with connect(f'Data/ICE13/cWFT/{cell_size}/ice13_{ice13_idx:02d}_3b.db') as db3:
                    elatt_cwft = 0
                    for i in range(len(db3)):
                        row = db3.get(id=i+1)
                        if 'DeltaMP2' in method:
                            elatt_cwft += (row.data['CCSD(T) Canonical DZ'] - row.data['MP2 Canonical DZ'] + row.data['MP2 Canonical CBS'])*(row.data['count'])
                        else:
                            elatt_cwft += row.data[f'{method} CBS']*(row.data['count'])
            method_elatt_dict['3B'] = elatt_cwft/num_monomers
            method_elatt_dict['Total'] = method_elatt_dict['1B'] + method_elatt_dict['2B'] + method_elatt_dict['3B']
            system_all_methods_elatt_dict[method] = method_elatt_dict.copy()

        ice13_final_corr_elatt_dict[cell_size][ice13_idx] = system_all_methods_elatt_dict


### DeltaMP2 (or Composite), LAF and Canonical approaches

In [36]:
ice13_final_corr_elatt_dict_unitcell = {}
ice13_final_corr_elatt_dict_mediumcell = {}
for ice13_idx in ice13_final_corr_elatt_dict['AFcell']:
    ice13_final_corr_elatt_dict_mediumcell[(ice13_systems[ice13_idx])] = {(mbe_type,method.split()[1]): ice13_final_corr_elatt_dict['AFcell'][ice13_idx][method][mbe_type] / kjmol for mbe_type in ['1B','2B','3B','Total'] for method in ['CCSD(T) Canonical','CCSD(T) LAF','CCSD(T) DeltaMP2'] }

for ice13_idx in ice13_final_corr_elatt_dict['Unitcell']:
    if ice13_idx == 7:
        continue
    # for mbe_type in ['1B','2B','3B','Total']:
    ice13_final_corr_elatt_dict_unitcell[(ice13_systems[ice13_idx])] = {(mbe_type,method.split()[1]): ice13_final_corr_elatt_dict['Unitcell'][ice13_idx][method][mbe_type] / kjmol for mbe_type in ['1B','2B','3B','Total'] for method in ['CCSD(T) Canonical','CCSD(T) LAF','CCSD(T) DeltaMP2'] }


# Add MAD row
ice13_final_corr_elatt_dict_unitcell['MAD'] = {}
for mbe_type in ['1B','2B','3B','Total']:
    for method in ['CCSD(T) Canonical','CCSD(T) LAF','CCSD(T) DeltaMP2']:
        ice13_final_corr_elatt_dict_unitcell['MAD'][(mbe_type,method.split()[1])] = np.mean([abs(ice13_final_corr_elatt_dict_unitcell[x][(mbe_type,method.split()[1])] - ice13_final_corr_elatt_dict_unitcell[x][(mbe_type,'Canonical')]) for x in ice13_final_corr_elatt_dict_unitcell.keys() if x != 'MAD'])
ice13_final_corr_elatt_dict_mediumcell['MAD'] = {}
for mbe_type in ['1B','2B','3B','Total']:
    for method in ['CCSD(T) Canonical','CCSD(T) LAF','CCSD(T) DeltaMP2']:
        ice13_final_corr_elatt_dict_mediumcell['MAD'][(mbe_type,method.split()[1])] = np.mean([abs(ice13_final_corr_elatt_dict_mediumcell[x][(mbe_type,method.split()[1])] - ice13_final_corr_elatt_dict_mediumcell[x][(mbe_type,'Canonical')]) for x in ice13_final_corr_elatt_dict_mediumcell.keys() if x != 'MAD'])


methods_df = pd.DataFrame(ice13_final_corr_elatt_dict_unitcell).T
# To nearest 1 d.p.
methods_df = methods_df.round(1)
# Write to latex table
display(methods_df)
# Convert to LaTeX table
convert_df_to_latex_input(
    df = methods_df,
    start_input = '\\begin{table}[h]\n',
    label = "tab:canonical_cheaper",
    caption = r"Comparison of various approaches [Canonical (Can.), LAF and Composite (Comp.) described in the text] for computing the lattice energy of several ice phases in their small-sized unit cells.",
    replace_input= {
        'DeltaMP2':r'Comp.',
        r'{Total} \\': r'{Total} \\ \cmidrule(lr){2-4} \cmidrule(lr){5-7} \cmidrule(lr){8-10} \cmidrule(lr){11-13}',
        r'MAD': r'\midrule MAD',
        r'Canonical' : r'Can.'
        },
    float_fmt="%.1f",
    filename = "Tables/SI_Table-Unitcell_Canonical_Cheaper.tex")

methods_df_medium = pd.DataFrame(ice13_final_corr_elatt_dict_mediumcell).T
# To nearest 1 d.p.
methods_df_medium = methods_df_medium.round(1)
# Write to latex table
display(methods_df_medium)
# Convert to LaTeX table
convert_df_to_latex_input(
    df = methods_df_medium,
    start_input = '\\begin{table}[h]\n',
    label = "tab:canonical_cheaper_medium",
    caption = r"Comparison of various approaches [Canonical (Can.), LAF and Composite (Comp.) described in the text] for computing the lattice energy of several ice phases in their medium-sized unit cells.",
    replace_input= {
        'DeltaMP2':r'Comp.',
        r'{Total} \\': r'{Total} \\ \cmidrule(lr){2-4} \cmidrule(lr){5-7} \cmidrule(lr){8-10} \cmidrule(lr){11-13}',
        r'MAD': r'\midrule MAD',
        r'Canonical' : r'Can.'
    },
    float_fmt="%.1f",
    filename = "Tables/SI_Table-Mediumcell_Canonical_Cheaper.tex")

1B                      2B                       3B                \
     Canonical  LAF DeltaMP2 Canonical   LAF DeltaMP2 Canonical  LAF DeltaMP2   
Ih        -8.0 -8.0     -8.1     -24.0 -24.0    -23.7       0.1  0.4      0.1   
II        -7.1 -7.1     -7.2     -26.7 -26.5    -26.4       1.7  1.8      1.8   
III       -8.0 -8.0     -8.1     -26.0 -25.7    -25.6       0.7  0.8      0.7   
IV        -7.1 -7.1     -7.1     -27.6 -27.4    -27.3       1.5  1.5      1.5   
VI        -6.8 -6.8     -6.8     -29.0 -28.8    -28.7       1.9  2.0      2.0   
VIII      -5.3 -5.3     -5.3     -31.0 -30.9    -30.7       2.5  1.9      2.6   
IX        -7.5 -7.5     -7.6     -25.1 -24.8    -24.8       1.1  0.6      1.1   
XI        -8.6 -8.6     -8.6     -24.6 -24.6    -24.2       0.0 -0.0      0.0   
XIII      -6.9 -6.9     -6.9     -27.0 -26.7    -26.7       1.5  1.3      1.5   
XIV       -6.6 -6.6     -6.7     -28.1 -27.7    -27.8       1.9  1.8      1.9   
XV        -6.5 -6.5     -6.6     -28.4 -28.2    -28.1       1.9  1.8      1.9   
XVII      -8.4 -8.4     -8.4     -23.5 -23.5    -23.1       0.3  0.4      0.3   
MAD        0.0  0.0      0.0       0.0   0.2      0.3       0.0  0.2      0.0   

         Total                 
     Canonical   LAF DeltaMP2  
Ih       -31.9 -31.6    -31.7  
II       -32.1 -31.8    -31.8  
III      -33.3 -32.9    -33.0  
IV       -33.2 -33.0    -32.9  
VI       -33.8 -33.5    -33.5  
VIII     -33.8 -34.3    -33.4  
IX       -31.6 -31.7    -31.3  
XI       -33.1 -33.2    -32.8  
XIII     -32.4 -32.3    -32.1  
XIV      -32.9 -32.6    -32.5  
XV       -33.1 -33.0    -32.8  
XVII     -31.6 -31.5    -31.3  
MAD        0.0   0.3      0.3

1B                      2B                       3B                \
     Canonical  LAF DeltaMP2 Canonical   LAF DeltaMP2 Canonical  LAF DeltaMP2   
Ih        -8.6 -8.6     -8.6     -23.8 -23.9    -23.4       0.1  0.4      0.1   
III       -7.5 -7.5     -7.6     -25.5 -25.2    -25.1       0.6  0.8      0.6   
IV        -7.0 -7.0     -7.0     -27.3 -27.0    -27.0       1.3  1.1      1.3   
VII       -5.6 -5.6     -5.6     -31.8 -31.7    -31.5       3.2  3.1      3.2   
XVII      -8.4 -8.4     -8.4     -23.4 -23.5    -23.0       0.2  0.4      0.2   
MAD        0.0  0.0      0.0       0.0   0.2      0.3       0.0  0.2      0.0   

         Total                 
     Canonical   LAF DeltaMP2  
Ih       -32.2 -32.0    -31.9  
III      -32.4 -31.9    -32.1  
IV       -33.0 -32.9    -32.7  
VII      -34.2 -34.2    -33.9  
XVII     -31.5 -31.5    -31.2  
MAD        0.0   0.2      0.3

## Periodic HF analysis

In [ ]:
# Analyse periodic HF calculations
ice13_final_hf_elatt_dict = {
    'AFsupercell': {y:0 for y in [1,3,4,6,7,17]},
    'AFcell': {y:0 for y in [1,3,4,7,17]},
    'Unitcell': {y:0 for y in ice13_systems.keys()}
}

# The overall lattice energy of the monomer is calculated from the HF calculations
ice13_final_elatt_dict = {
    'AFsupercell': {y:0 for y in [1,3,4,6,7,17]},
    'AFcell': {y:0 for y in [1,3,4,7,17]},
    'Unitcell': {y:0 for y in ice13_systems.keys()},
    'Final': {y:0 for y in ice13_systems.keys()},
    'Final CCSD': {y:0 for y in ice13_systems.keys()},
    'Final MP2': {y:0 for y in ice13_systems.keys()}
}

for cell_size in ['AFcell','Unitcell','AFsupercell']:
    molecule = read(f'Data/HF/ICE13/Molecule/01-Hard_PAW_Default/OUTCAR_HF')
    molecule_ene = []
    for calc_type in ['01-Hard_PAW_Default','02-Standard_PAW_Default','03-Standard_PAW_High']:
        molecule_ene += [get_energy(f'Data/HF/ICE13/Molecule/{calc_type}/OUTCAR_HF',code='vasp')]

    for ice13_idx in ice13_final_hf_elatt_dict[cell_size].keys():
        solid = read(f'Data/HF/ICE13/{cell_size}/01-Hard_PAW_Default/{ice13_idx:02d}/OUTCAR_HF')
        num_monomers = int(len(solid)/len(molecule))
        solid_ene = []
        for calc_type in ['01-Hard_PAW_Default','02-Standard_PAW_Default','03-Standard_PAW_High']:
            solid_ene += [get_energy(f'Data/HF/ICE13/{cell_size}/{calc_type}/{ice13_idx:02d}/OUTCAR_HF',code='vasp')]

        elatt_ene = np.array(solid_ene)/num_monomers - molecule_ene
        elatt = (elatt_ene[0] + elatt_ene[2] - elatt_ene[1])*1000
        ice13_final_hf_elatt_dict[cell_size][ice13_idx] = elatt


for cell_size in ['AFsupercell','AFcell','Unitcell']:
    for ice13_idx in ice13_final_corr_elatt_dict[cell_size].keys():
        if cell_size == 'AFsupercell':
            ice13_final_elatt_dict[cell_size][ice13_idx] = (ice13_final_hf_elatt_dict['AFsupercell'][ice13_idx] + ice13_final_corr_elatt_dict[cell_size][ice13_idx]['CCSD(T) LAF']['Total'])/kjmol
        # elif cell_size == 'AFsupercell':
        #     ice13_final_elatt_dict[cell_size][ice13_idx] = (ice13_final_hf_elatt_dict['Unitcell'][ice13_idx] + ice13_final_corr_elatt_dict[cell_size][ice13_idx]['CCSD(T) LAF']['Total'])/kjmol
        else:
            ice13_final_elatt_dict[cell_size][ice13_idx] = (ice13_final_hf_elatt_dict[cell_size][ice13_idx] + ice13_final_corr_elatt_dict[cell_size][ice13_idx]['CCSD(T) Canonical']['Total'])/kjmol

# Final lattice energies
for ice13_idx in ice13_final_elatt_dict['Final'].keys():
    if ice13_idx == 7:
        ice13_final_elatt_dict['Final'][ice13_idx] = ice13_final_elatt_dict['AFcell'][ice13_idx]
        ice13_final_elatt_dict['Final CCSD'][ice13_idx] = (ice13_final_hf_elatt_dict['AFcell'][ice13_idx] + ice13_final_corr_elatt_dict['AFcell'][ice13_idx]['CCSD Canonical']['Total'])/kjmol
        ice13_final_elatt_dict['Final MP2'][ice13_idx] = (ice13_final_hf_elatt_dict['AFcell'][ice13_idx] + ice13_final_corr_elatt_dict['AFcell'][ice13_idx]['MP2 Canonical']['Total'])/kjmol
    else:
        ice13_final_elatt_dict['Final'][ice13_idx] = ice13_final_elatt_dict['Unitcell'][ice13_idx]
        ice13_final_elatt_dict['Final CCSD'][ice13_idx] = (ice13_final_hf_elatt_dict['Unitcell'][ice13_idx] + ice13_final_corr_elatt_dict['Unitcell'][ice13_idx]['CCSD Canonical']['Total'])/kjmol
        ice13_final_elatt_dict['Final MP2'][ice13_idx] = (ice13_final_hf_elatt_dict['Unitcell'][ice13_idx] + ice13_final_corr_elatt_dict['Unitcell'][ice13_idx]['MP2 Canonical']['Total'])/kjmol